# Task 1: Mapping the Review Landscape

This notebook initiates the Yelp Capstone project by establishing a foundational **evidence layer** through the analysis of restaurant reviews within a specific cuisine—in this case, **Mexican**.

### The Core Objective
How do diners describe their experiences when they are delighted versus when they are disappointed? To answer this, we move beyond simple star ratings to extract the underlying **themes, dishes, and service patterns** that define the Mexican restaurant category.

### Our Three-Paradigm Approach
To ensure our findings are robust, we analyze the text through three distinct data mining lenses:
1.  **The Frequency Paradigm (N-grams):** We start with the obvious. What are the literal phrases (like "carne asada" or "slow service") that appear most often?
2.  **The Probability Paradigm (LDA):** We look for latent themes. Even if words vary, can we find the hidden 'topics'—such as *Ambiance* or *Authenticity*—that reviews naturally cluster into?
3.  **The Geometry Paradigm (BERTopic):** We use modern AI embeddings to find semantic meaning. This allows us to group reviews that *mean* the same thing even if they use entirely different vocabularies.

### Why This Matters
The artifacts produced here—including a cleaned review subset and thematic summaries—are not just for this report. They serve as the direct inputs for **Task 2 (Cuisine Mapping)**, **Task 3 (Phrase Discovery)**, and **Task 4 (Search and Ranking)**. By the end of this notebook, we will have a data-driven signature of what makes or breaks a Mexican restaurant experience.

## Table of Contents

1. [Runtime and Package Setup](#0.-Runtime-and-package-setup)
2. [Dataset Acquisition and Loading](#2.-Dataset-acquisition-and-loading)
3. [Build Restaurant/Cuisine Subset](#3.-Build-the-restaurant/cuisine-subset)
4. [Exploratory Data Analysis (EDA)](#4.-Exploratory-data-analysis)
5. [Text Cleaning Helpers](#5.-Text-cleaning-helpers)
6. [Approach 1: N-gram Frequency Analysis](#6.-Approach-1-%E2%80%94-N-gram-frequency-analysis)
7. [Approach 2: Classical Topic Modeling (LDA)](#7.-Approach-2-%E2%80%94-Classical-Topic-Modeling-With-LDA)
8. [Approach 3: Modern Embedding Analysis (BERTopic)](#8.-Approach-3-%E2%80%94-Modern-Embedding/Topic-Clustering-Analysis)
9. [Cross-Method Comparison](#9.-Compare-the-three-approaches)
10. [Report Deliverables and Validation](#10.-Required-Report-Deliverables)

# Task 1 - Data Set Exploration and Thematic Analysis

In this notebook, you will create the first evidence layer of the Yelp capstone by analyzing restaurant reviews from a focused cuisine or category subset.

## Learning goals
1. Build a reproducible subset of restaurant reviews.
2. Describe the subset with clear EDA evidence.
3. Compare themes using n-grams, classical LDA, and a modern topic method.
4. Compare high-rated vs. low-rated review language and themes.
5. Save artifacts that will be reused in later tasks.


## *Smoke Test mode*

*Use `smoke_test` mode only to verify that your notebook runs end-to-end when real data is unavailable. Base your final conclusions on real assignment data and clearly label any fallback method outputs.*


## Required artifact contract for Task 1

Your completed notebook should produce these core artifacts under `artifacts/task1/`:

- `review_subset.csv` — the bounded restaurant/cuisine review subset.
- `task1_overall_top_ngrams.csv` — frequency-based thematic evidence from n-gram analysis.
- `task1_lda_topics.csv` — LDA topic-model outputs (domain-specific stopwords).
- `task1_modern_topic_diagnostics.csv` — BERTopic or clearly labeled fallback diagnostics.
- `theme_comparison_summary.csv` — final summary comparing all three methods.

Optional figures are welcome, but these tables are the minimum handoff artifacts for grading.

## 0. Runtime and package setup

### Local installation and run

You can download and run this notebook on your personal computer, in which case it will probably have access to a faster CPU than what you would get on Google Colab. We highly recommend you download and run this notebook on your own computer.

[Jupyter Installation Instructions](https://docs.jupyter.org/en/latest/install.html)

### Google Colab

You can also opt to run it on Google Colab, but be ware that Colab turns off the underlying cloud resources running your notebook after a set amount of time, at which point you have to rerun all cells (including `pip install` and dataset downloads).


- Use **Runtime → Change runtime type → Python 3**.
- A GPU is optional. BERTopic/SentenceTransformers runs faster with GPU, but Task 1 should remain feasible on CPU with a smaller sample.
- Start with modest sample sizes. Students can increase them after the pipeline runs.


In [ ]:
# Optional: install packages in Colab.
# Keep this False for local smoke tests or controlled course environments.
INSTALL_EXTRA_PACKAGES = False

if INSTALL_EXTRA_PACKAGES:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "kagglehub", "bertopic", "sentence-transformers", "umap-learn", "hdbscan"
    ])

In [ ]:
import os
import re
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_colwidth", 120)


In [ ]:
# my code
# this cell is to ensure the notebook runs on any machine / Colab

import os
from pathlib import Path

# create a function to help to find the project root
def find_project_root(start: Path) -> Path:
    """Walk up from `start` until we find the repo root (contains artifacts/ or notebooks/)."""
    for candidate in [start, *start.parents]:
        if (candidate / "artifacts").is_dir() or (candidate / "notebooks").is_dir():
            return candidate
    return start


# Resolve the project root portably so the notebook runs on any machine / Colab:
# explicit override via the DMC_PROJECT_ROOT environment variable, else
# auto-detect by walking up from the current working directory.
env_root=os.environ.get("DMC_PROJECT_ROOT", "").strip()
if env_root:
    PROJECT_ROOT=Path(env_root).expanduser().resolve()
else:
    PROJECT_ROOT=find_project_root(Path.cwd().resolve())
# change the working directory to the project root
os.chdir(PROJECT_ROOT)
# set the environment variables
os.environ.setdefault("TASK1_DATA_SOURCE", "manual")
os.environ.setdefault("DMC_RAW_DATA_DIR", "data/raw")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", os.getcwd())
print("DATA_SOURCE will be:", os.environ.get("TASK1_DATA_SOURCE"))
print("DMC_RAW_DATA_DIR:", os.environ.get("DMC_RAW_DATA_DIR"))

## 1. Configuration

In this notebook, we use the Yelp **Mexican restaurant reviews** as the primary worked example. This choice is motivated by the availability of a sufficiently large subset in the public Yelp dataset and its natural connection to subsequent tasks notebooks involving cuisines, dishes, sentiment analysis, and restaurant recommendation.

You may instead select another cuisine or category. If you do so, you must justify your choice and verify that both high-rated and low-rated subsets contain enough reviews to enable a meaningful comparative analysis.

In [ ]:
# Core assignment configuration
TARGET_CATEGORY = "Mexican"      # Instructor worked example; students may choose another cuisine/category.
REQUIRE_RESTAURANTS = True        # Keep the capstone focused on restaurant reviews.

# Data source options:
#   "kaggle_latest"  -> kagglehub.dataset_download("yelp-dataset/yelp-dataset")
#   "manual"         -> manually set DATA_DIR to a directory containing Yelp JSON files
#   "smoke_test"     -> synthetic Yelp-shaped data used only for validation without internet
DATA_SOURCE = os.environ.get("DMC_DATASET_MODE", os.environ.get("TASK1_DATA_SOURCE", "smoke_test"))
DATA_DIR = os.environ.get("TASK1_DATA_DIR", "")
if DATA_SOURCE == "smoke_test":
    DATA_DIR = os.environ.get("DMC_SMOKE_DATA_DIR", DATA_DIR)
elif DATA_SOURCE == "manual":
    DATA_DIR = os.environ.get("DMC_RAW_DATA_DIR", DATA_DIR)

# Runtime controls
MAX_BUSINESSES = None             # Set to a small integer for debugging, otherwise None.
MAX_REVIEWS = 50_000              # Enough for meaningful analysis but feasible in Colab.
CHUNKSIZE = 250_000               # For streaming JSON lines.

# Sampling controls. We collect a larger capped candidate pool, then sample the final
# analysis subset so one chain or high-volume business cannot dominate the themes.
REVIEW_CANDIDATE_MULTIPLIER = int(os.environ.get("TASK1_REVIEW_CANDIDATE_MULTIPLIER", "3"))
MAX_REVIEW_CANDIDATES = int(os.environ.get("TASK1_MAX_REVIEW_CANDIDATES", str(MAX_REVIEWS * REVIEW_CANDIDATE_MULTIPLIER)))
MAX_CANDIDATE_REVIEWS_PER_BUSINESS = int(os.environ.get("TASK1_MAX_CANDIDATE_REVIEWS_PER_BUSINESS", "120"))
MAX_FINAL_REVIEWS_PER_BUSINESS = int(os.environ.get("TASK1_MAX_FINAL_REVIEWS_PER_BUSINESS", "60"))

CACHE_DIR = Path(os.environ.get("DMC_CACHE_DIR", "task1_cache"))
ARTIFACT_DIR = Path(os.environ.get("DMC_ARTIFACT_DIR", "artifacts/task1"))
CACHE_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

# Modeling controls
NGRAM_SAMPLE_N = 30_000
LDA_SAMPLE_N = 12_000
MODERN_SAMPLE_N = 4_000           # Backward-compatible default for modern-topic sample sizes.
MODERN_TARGET_SAMPLE_N = int(os.environ.get("TASK1_MODERN_TARGET_SAMPLE_N", "3_000"))
MODERN_CROSS_CUISINE_SAMPLE_N = int(os.environ.get("TASK1_MODERN_CROSS_CUISINE_SAMPLE_N", str(MODERN_SAMPLE_N)))
MODERN_TOPIC_CATEGORIES = [
    "Mexican", "Italian", "Chinese", "Thai", "Indian", "Pizza", "Sushi"
]
MODERN_REVIEWS_PER_CATEGORY = max(100, MODERN_SAMPLE_N // len(MODERN_TOPIC_CATEGORIES))
N_LDA_TOPICS = 8
REFINED_LDA_NGRAM_RANGE = (1, 2)
REFINED_LDA_MAX_FEATURES = 8_000
N_MODERN_TOPICS_FALLBACK = 8

print("Configuration")
print("-------------")
print("DATA_SOURCE:", DATA_SOURCE)
print("TARGET_CATEGORY:", TARGET_CATEGORY)
print("MAX_REVIEWS:", MAX_REVIEWS)


### Configuration checklist

Before running the analysis, make sure your setup is bounded and reproducible:
- Choose a target cuisine/category and keep the decision explicit.
- Keep sampling limits visible (`MAX_REVIEWS`, per-business caps, and sample sizes).
- Keep paths configurable via environment variables.
- Keep random seeds fixed so results are reproducible.

## 2. Dataset acquisition and loading

The course uses the current public Kaggle Yelp dataset, either downloaded through `kagglehub` or provided as a local manual copy.

The loader is deliberately written in a streaming style. You should not load the full review file into memory.

In [ ]:
def download_or_locate_dataset(data_source: str, data_dir: str = "") -> Path:
    """Return a directory that contains Yelp JSON files."""
    if data_source == "manual":
        if not data_dir:
            raise ValueError("For DATA_SOURCE='manual', set DATA_DIR to the dataset directory.")
        return Path(data_dir)

    if data_source == "kaggle_latest":
        try:
            import kagglehub
        except ImportError as e:
            raise ImportError(
                "kagglehub is not installed. In Colab, set INSTALL_EXTRA_PACKAGES=True or run: pip install kagglehub. "
                "Alternatively set DATA_SOURCE='manual' and point DATA_DIR to an extracted Yelp dataset."
            ) from e
        path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")
        return Path(path)

    if data_source == "smoke_test":
        smoke_dir = Path(data_dir) if data_dir else CACHE_DIR / "smoke_yelp"
        if not smoke_dir.exists() or not list(smoke_dir.glob("*.json")):
            return create_smoke_test_dataset(smoke_dir)
        return smoke_dir

    raise ValueError(f"Unknown DATA_SOURCE: {data_source}")


def find_yelp_file(dataset_dir: Path, kind: str) -> Path:
    """Find a Yelp JSON-lines file recursively by kind: business, review, user, etc."""
    candidates = []
    for p in dataset_dir.rglob("*.json"):
        name = p.name.lower()
        if kind.lower() in name:
            candidates.append(p)
    if not candidates:
        # Some extracted archives use .json files nested oddly, so show useful debugging info.
        sample = list(dataset_dir.rglob("*"))[:25]
        raise FileNotFoundError(f"Could not find a JSON file containing '{kind}' under {dataset_dir}. Sample: {sample}")
    # Prefer academic dataset naming if present.
    candidates = sorted(candidates, key=lambda p: ("academic" not in p.name.lower(), len(str(p))))
    return candidates[0]


### 2.1 Smoke-test fixture used for local validation only

This cell creates a small Yelp-shaped dataset. It is not used in the real assignment. It exists so the notebook can be executed end-to-end in an offline environment.

In [ ]:
def create_smoke_test_dataset(out_dir: Path, n_businesses: int = 40, n_reviews: int = 1200) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    business_path = out_dir / "yelp_academic_dataset_business.json"
    review_path = out_dir / "yelp_academic_dataset_review.json"

    rng = np.random.default_rng(RANDOM_SEED)
    cities = ["Phoenix", "Las Vegas", "Toronto", "Madison"]
    cuisines = ["Mexican", "Italian", "Chinese", "Thai"]

    businesses = []
    for i in range(n_businesses):
        cuisine = cuisines[i % len(cuisines)]
        cats = f"Restaurants, {cuisine}, Food"
        businesses.append({
            "business_id": f"biz_{i:04d}",
            "name": f"{cuisine} Bistro {i}",
            "city": cities[i % len(cities)],
            "state": "AZ",
            "stars": float(rng.choice([2.5, 3.0, 3.5, 4.0, 4.5])),
            "review_count": int(rng.integers(20, 800)),
            "categories": cats,
        })

    positive_phrases = [
        "the tacos were fresh and the salsa was bright",
        "excellent mole and friendly service",
        "great burrito with flavorful carne asada",
        "the patio was lively and the margaritas were excellent",
        "delicious enchiladas and warm tortillas",
    ]
    negative_phrases = [
        "the tacos were cold and the service was slow",
        "the burrito was bland and the rice was dry",
        "overpriced meal with a long wait",
        "the salsa tasted watery and the room was noisy",
        "the enchiladas were soggy and disappointing",
    ]
    neutral_phrases = [
        "we visited for dinner on a weekday",
        "the restaurant is located near a busy street",
        "parking was available behind the building",
    ]

    mexican_ids = [b["business_id"] for b in businesses if "Mexican" in b["categories"]]
    all_ids = [b["business_id"] for b in businesses]
    reviews = []
    for i in range(n_reviews):
        # Bias toward Mexican restaurants so the target subset is not tiny.
        bid = str(rng.choice(mexican_ids if rng.random() < 0.65 else all_ids))
        stars = int(rng.choice([1, 2, 3, 4, 5], p=[0.13, 0.14, 0.13, 0.30, 0.30]))
        if stars >= 4:
            text = str(rng.choice(positive_phrases))
        elif stars <= 2:
            text = str(rng.choice(negative_phrases))
        else:
            text = str(rng.choice(neutral_phrases))
        reviews.append({
            "review_id": f"rev_{i:06d}",
            "user_id": f"user_{int(rng.integers(0, 300)):04d}",
            "business_id": bid,
            "stars": stars,
            "date": f"{int(rng.integers(2015, 2022))}-{int(rng.integers(1,13)):02d}-{int(rng.integers(1,28)):02d}",
            "text": text + ". " + str(rng.choice(neutral_phrases)),
            "useful": int(rng.integers(0, 20)),
            "funny": int(rng.integers(0, 5)),
            "cool": int(rng.integers(0, 5)),
        })

    with business_path.open("w", encoding="utf-8") as f:
        for row in businesses:
            f.write(json.dumps(row) + "\n")
    with review_path.open("w", encoding="utf-8") as f:
        for row in reviews:
            f.write(json.dumps(row) + "\n")
    return out_dir


In [ ]:
dataset_dir = download_or_locate_dataset(DATA_SOURCE, DATA_DIR)
business_path = find_yelp_file(dataset_dir, "business")
review_path = find_yelp_file(dataset_dir, "review")

print("Dataset directory:", dataset_dir)
print("Business file:", business_path)
print("Review file:", review_path)


## 3. Build the restaurant/cuisine subset
To build the subset:

- Choose restaurants whose categories include `Restaurants` and `Mexican`.
- Stream reviews and keep only reviews for those businesses.
- Label reviews as `high` if stars ≥ 4 and `low` if stars ≤ 2. Neutral reviews are retained for overall EDA but excluded from the high/low thematic comparison.

### Sampling design note: avoid high-volume business domination

If you were to naively sample data, a few high-volume businesses can dominate the subset, causing themes and topics to reflect those restaurants more than the broader cuisine/category.

To solve this issue, we use a capped, deterministic review sampling strategy instead of taking the first matching reviews from the streamed Yelp file. The candidate-pool and per-business caps keep runtime bounded while making the final review subset more representative for EDA, topic modeling, and downstream artifact handoff.

This is fundamentally a modeling decision rather than a mere engineering detail. The sampling frame determines the vocabulary, topic prevalence, high-versus-low comparisons, and all subsequent artifacts that reuse the Task 1 review subset.

In [ ]:
def normalize_categories(x) -> str:
    if isinstance(x, list):
        return ", ".join(map(str, x))
    if pd.isna(x):
        return ""
    return str(x)


def load_target_businesses(business_path: Path, target_category: str, require_restaurants: bool = True, max_businesses=None) -> pd.DataFrame:
    usecols = None  # pandas json reader does not always support usecols consistently for lines JSON.
    businesses = []
    for chunk in pd.read_json(business_path, lines=True, chunksize=100_000):
        needed = [c for c in ["business_id", "name", "city", "state", "stars", "review_count", "categories"] if c in chunk.columns]
        temp = chunk[needed].copy()
        temp["categories_text"] = temp.get("categories", "").apply(normalize_categories)
        cat = temp["categories_text"].str.lower()
        mask = cat.str.contains(re.escape(target_category.lower()), na=False)
        if require_restaurants:
            mask &= cat.str.contains("restaurant", na=False)
        keep = temp.loc[mask].copy()
        if not keep.empty:
            businesses.append(keep)
        if max_businesses is not None and sum(len(x) for x in businesses) >= max_businesses:
            break

    if not businesses:
        raise ValueError(f"No businesses found for target_category={target_category!r}.")

    out = pd.concat(businesses, ignore_index=True).drop_duplicates("business_id")
    if max_businesses is not None:
        out = out.head(max_businesses)
    return out


target_business_df = load_target_businesses(
    business_path,
    TARGET_CATEGORY,
    require_restaurants=REQUIRE_RESTAURANTS,
    max_businesses=MAX_BUSINESSES,
)

print(f"Target businesses: {len(target_business_df):,}")
display(target_business_df.head())


In [ ]:
def _cap_reviews_per_business(df: pd.DataFrame, cap: int, seed: int = RANDOM_SEED) -> pd.DataFrame:
    if cap <= 0 or df.empty:
        return df.iloc[0:0].copy()
    pieces = []
    for _, group in df.groupby("business_id", sort=False):
        if len(group) <= cap:
            pieces.append(group)
        else:
            pieces.append(group.sample(n=cap, random_state=seed))
    if not pieces:
        return df.iloc[0:0].copy()
    return pd.concat(pieces, ignore_index=True)


def load_reviews_for_businesses(
    review_path: Path,
    business_ids: set,
    max_reviews: int = 50_000,
    chunksize: int = 250_000,
    max_candidate_reviews: int = MAX_REVIEW_CANDIDATES,
    max_candidate_per_business: int = MAX_CANDIDATE_REVIEWS_PER_BUSINESS,
    max_final_per_business: int = MAX_FINAL_REVIEWS_PER_BUSINESS,
) -> tuple[pd.DataFrame, dict]:
    pieces = []
    per_business_counts = Counter()
    n_kept = 0
    keep_cols = ["review_id", "user_id", "business_id", "stars", "date", "text", "useful", "funny", "cool"]

    for chunk_i, chunk in enumerate(pd.read_json(review_path, lines=True, chunksize=chunksize), start=1):
        part = chunk[chunk["business_id"].isin(business_ids)].copy()
        if part.empty:
            continue
        cols = [c for c in keep_cols if c in part.columns]
        part = part[cols]

        capped_parts = []
        for bid, group in part.groupby("business_id", sort=False):
            remaining = max_candidate_per_business - per_business_counts[bid]
            if remaining <= 0:
                continue
            take = group.head(remaining).copy()
            per_business_counts[bid] += len(take)
            capped_parts.append(take)

        if capped_parts:
            kept = pd.concat(capped_parts, ignore_index=True)
            pieces.append(kept)
            n_kept += len(kept)
            print(f"chunk {chunk_i}: kept {len(kept):,}; candidate cumulative {n_kept:,}")
        if n_kept >= max_candidate_reviews:
            break

    if not pieces:
        raise ValueError("No reviews found for the selected businesses.")

    candidate_reviews = pd.concat(pieces, ignore_index=True)
    candidate_reviews["date"] = pd.to_datetime(candidate_reviews["date"], errors="coerce")
    candidate_reviews["stars"] = pd.to_numeric(candidate_reviews["stars"], errors="coerce")
    candidate_reviews = candidate_reviews.dropna(subset=["business_id", "stars", "text"]).copy()
    candidate_reviews["rating_group"] = np.select(
        [candidate_reviews["stars"] >= 4, candidate_reviews["stars"] <= 2],
        ["high", "low"],
        default="neutral",
    )

    capped_pool = _cap_reviews_per_business(candidate_reviews, max_final_per_business, seed=RANDOM_SEED)
    if len(capped_pool) > max_reviews:
        reviews = capped_pool.sample(n=max_reviews, random_state=RANDOM_SEED).reset_index(drop=True)
    else:
        reviews = capped_pool.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

    candidate_business_counts = candidate_reviews["business_id"].value_counts()
    final_business_counts = reviews["business_id"].value_counts()
    diagnostics = {
        "sampling_strategy": "capped_candidate_pool_then_deterministic_final_sample",
        "target_businesses": len(business_ids),
        "candidate_reviews_collected": int(len(candidate_reviews)),
        "candidate_businesses": int(candidate_reviews["business_id"].nunique()),
        "max_candidate_reviews_per_business": int(max_candidate_per_business),
        "final_reviews": int(len(reviews)),
        "final_businesses": int(reviews["business_id"].nunique()),
        "max_final_reviews_per_business_setting": int(max_final_per_business),
        "observed_max_final_reviews_per_business": int(final_business_counts.max()) if not final_business_counts.empty else 0,
        "median_final_reviews_per_business": float(final_business_counts.median()) if not final_business_counts.empty else 0.0,
        "candidate_rating_group_counts": candidate_reviews["rating_group"].value_counts().to_dict(),
        "final_rating_group_counts": reviews["rating_group"].value_counts().to_dict(),
        "candidate_top_business_review_count": int(candidate_business_counts.max()) if not candidate_business_counts.empty else 0,
    }
    return reviews, diagnostics


business_ids = set(target_business_df["business_id"])
review_df, review_sampling_diagnostics = load_reviews_for_businesses(
    review_path,
    business_ids,
    max_reviews=MAX_REVIEWS,
    chunksize=CHUNKSIZE,
)

# Add selected business metadata for EDA.
business_meta_cols = [c for c in ["business_id", "name", "city", "state", "stars", "review_count", "categories_text"] if c in target_business_df.columns]
review_df = review_df.merge(target_business_df[business_meta_cols], on="business_id", how="left", suffixes=("", "_business"))

review_sampling_summary = pd.DataFrame([
    {"metric": key, "value": json.dumps(value) if isinstance(value, dict) else value}
    for key, value in review_sampling_diagnostics.items()
])
review_sampling_summary.to_csv(ARTIFACT_DIR / "task1_sampling_diagnostics.csv", index=False)
top_business_sampling = (
    review_df.groupby(["business_id", "name"], dropna=False)
    .agg(reviews=("review_id", "count"), avg_stars=("stars", "mean"))
    .sort_values("reviews", ascending=False)
    .reset_index()
)
top_business_sampling.to_csv(ARTIFACT_DIR / "task1_sampling_top_businesses.csv", index=False)
display(review_sampling_summary)
display(top_business_sampling.head(15))

cache_base = CACHE_DIR / f"task1_{TARGET_CATEGORY.lower().replace(' ', '_')}_reviews"
try:
    cache_path = cache_base.with_suffix(".parquet")
    review_df.to_parquet(cache_path, index=False)
except Exception as e:
    # Parquet requires pyarrow or fastparquet. Fall back to pickle so the notebook still runs.
    print("Parquet unavailable; saving pickle cache instead:", repr(e))
    cache_path = cache_base.with_suffix(".pkl")
    review_df.to_pickle(cache_path)
print(f"Saved subset cache: {cache_path}")
print("Rows:", len(review_df))
display(review_df.head())
display(review_df["rating_group"].value_counts())


### Subset quality check

Before moving to modeling, verify that your subset can support comparison:
- enough total reviews for stable patterns,
- non-trivial counts in both `high` and `low` groups,
- a short rationale for why this cuisine/category is analytically interesting.

In [ ]:
# TODO (Category E — Guided pandas): Build a summary table of your review subset.
# Estimated time: ~0.5 h
#
# Expected input:
#   - target_business_df : DataFrame of businesses matching TARGET_CATEGORY
#   - review_df          : review DataFrame with columns business_id, stars, date, rating_group
#   - review_sampling_diagnostics : dict returned by load_reviews(); keys include
#       "sampling_strategy" and "observed_max_final_reviews_per_business"
#
# Expected output:
#   - subset_summary : two-column DataFrame with columns ["metric", "value"]
#   - Saved to       : ARTIFACT_DIR / "task1_subset_summary.csv"
#
# Metrics to include (use this exact order):
#   target_category, n_target_businesses, n_reviewed_businesses, n_reviews,
#   n_high, n_low, n_neutral, date_min, date_max,
#   sampling_strategy, max_final_reviews_per_business_observed
#
# Hints:
#   - n_reviewed_businesses = review_df["business_id"].nunique()
#   - n_high = (review_df["rating_group"] == "high").sum()  (cast to int)
#   - date_min: str(review_df["date"].min().date()) if review_df["date"].notna().any() else "NA"
subset_summary = pd.DataFrame({
    "metric": [
        "target_category", "n_target_businesses", "n_reviewed_businesses", "n_reviews",
        "n_high", "n_low", "n_neutral", "date_min", "date_max",
        "sampling_strategy", "max_final_reviews_per_business_observed",
    ],
    "value": [
        # TODO: replace each None with the correct expression
        # my code
        # target_category
        TARGET_CATEGORY, 
        # n_target_businesses                                             
        len(target_business_df),  
        # n_reviewed_businesses                                    
        review_df["business_id"].nunique(), 
        # n_reviews                          
        len(review_df),  

        # n_high
        int((review_df["rating_group"] == "high").sum()),   
        # n_low          
        int((review_df["rating_group"] == "low").sum()),  
        # n_neutral            
        int((review_df["rating_group"] == "neutral").sum()),  
        # date_min        
        str(review_df["date"].min().date()) if review_df["date"].notna().any() else "NA", 
        # date_max
        str(review_df["date"].max().date()) if review_df["date"].notna().any() else "NA", 
        # sampling_strategy 
        review_sampling_diagnostics["sampling_strategy"],    
        # max_final_reviews_per_business_observed           
        review_sampling_diagnostics["observed_max_final_reviews_per_business"],  
    ]
})
display(subset_summary)
subset_summary.to_csv(ARTIFACT_DIR / "task1_subset_summary.csv", index=False)

# --- Validation ---
assert list(subset_summary.columns) == ["metric", "value"], "Must have exactly columns: metric, value"
assert len(subset_summary) == 11, f"Expected 11 rows, got {len(subset_summary)}"
assert subset_summary["value"].notna().all(), "All values must be filled in (no None remaining)"
print("\u2713 subset_summary looks good:", len(subset_summary), "rows")


### Interpreting subset balance

Use your saved subset summary to verify that high/low comparisons are meaningful. If one rating group is very small, call that out and discuss how it limits interpretation.

## 4. Exploratory data analysis

You should include at least two meaningful EDA plots. Some suggested plots could be:

1. rating distribution;
2. review counts by year;
3. top cities in the selected subset.

In [ ]:
def save_current_figure(name: str):
    path = ARTIFACT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("saved", path)

# TODO (Category B — Classic visualization): Create three EDA plots for your review subset.
# Estimated time: ~1 h
#
# Expected input:
#   review_df with columns: stars (int 1-5), date (datetime, may have NaT), city (str, may be absent)
#   TARGET_CATEGORY : string used in plot titles
#   ARTIFACT_DIR    : Path object for saving figures
#
# Expected output — three plot files saved via save_current_figure():
#   "task1_rating_distribution.png"  — bar chart of star-rating counts
#   "task1_reviews_over_time.png"    — line chart of reviews per year (skip if no valid dates)
#   "task1_top_cities.png"           — horizontal bar chart of top-12 cities (skip if no city column)
#
# Steps:
#  1. Plot review_df["stars"].value_counts().sort_index() as a bar chart (figsize 7×4).
#     Add axis labels, title, call save_current_figure(), then plt.show() + plt.close().
#
#  2. If review_df["date"].notna().any():
#     Extract year with review_df.dropna(subset=["date"]).assign(year=...).
#     Count reviews per year, plot as a line with markers (figsize 9×4).
#     Call save_current_figure("task1_reviews_over_time.png").
#
#  3. If "city" in review_df.columns:
#     Take top-12 cities by count, sort ascending (for readability), horizontal bar (figsize 8×5).
#     Call save_current_figure("task1_top_cities.png").
#
# Hint: .value_counts().sort_index() vs .value_counts().sort_values() for correct axis ordering.

# --- your code here ---
# my code
# figure 01: image for rating distribution
# get the number of reviews for each star rating and sort them
number_of_views_for_rating_stars=review_df["stars"].value_counts().sort_index()
# create a bar chart(figure size 7*4) for the number of reviews for each star rating
plt.figure(figsize=(7, 4))
plt.bar(number_of_views_for_rating_stars.index.astype(str), number_of_views_for_rating_stars.values)
# add axis labels and title
plt.xlabel("Rating Stars")
# add y axis label
plt.ylabel("Number of Reviews")
# add title
plt.title(f"Rating Distribution — {TARGET_CATEGORY} Reviews")
# add x axis ticks
plt.xticks(number_of_views_for_rating_stars.index.astype(str))
# save the figure
save_current_figure("task1_rating_distribution.png")
# show the figure
plt.show()
# close the figure
plt.close()


# figure 02: image for reviews by year
# check if there exist valid dates
if_valid_dates_exist=review_df["date"].notna().any()
# if there exist valid dates
if if_valid_dates_exist:
    # remove the rows without date
    reviews_with_date=review_df.dropna(subset=["date"]).assign(year=lambda df: df["date"].dt.year)
    
    # get the number of reviews for each year
    number_of_reviews_per_year = reviews_with_date["year"].value_counts().sort_index()
    # create a line chart(figure size 9*4) for the number of reviews for each year
    plt.figure(figsize=(9, 4))
    plt.plot(number_of_reviews_per_year.index, number_of_reviews_per_year.values, marker="o")
    # add axis labels and title
    plt.xlabel("Year")
    # add y axis label
    plt.ylabel("Number of Reviews for Each Year")
    # add title
    plt.title(f"Reviews Over Time — {TARGET_CATEGORY} Reviews")
    # add grid
    plt.grid(True, alpha=0.5)
    # save the figure
    save_current_figure("task1_reviews_over_time.png")
    # show the figure
    plt.show()
    # close the figure
    plt.close()
# if there is no valid date
else:
    print("we can't find valid dates, so we can't plot the reviews by year")


# figure 03: image for top cities in the selected subset
# check if there exist city column
if_city_column_exist="city" in review_df.columns
# if there exist city column
if if_city_column_exist:
    # get the number of reviews for each city, get the top 12 and sort them
    number_of_city_review=review_df["city"].value_counts().head(12)
    # sort the number of reviews for each city from small to large
    top_12_cities_sorted=number_of_city_review.sort_values(ascending=True)
    # create a horizontal bar chart(figure size 8*5) for the number of reviews for each city
    plt.figure(figsize=(8, 5))
    # add the number of reviews for each city to the horizontal bar chart
    plt.barh(top_12_cities_sorted.index, top_12_cities_sorted.values)
    # add axis labels and title
    plt.xlabel("Number of Reviews")
    # add y axis label
    plt.ylabel("City")
    # add title
    plt.title(f"Top 12 Cities — {TARGET_CATEGORY} Reviews")
    # save the figure
    save_current_figure("task1_top_cities.png")
    # show the figure
    plt.show()
    # close the figure
    plt.close()
else:
    print("we can't find city column, so we can't plot the top cities")

# --- Validation ---
assert (ARTIFACT_DIR / "task1_rating_distribution.png").exists(), \
    "Rating distribution plot not saved — did you call save_current_figure()?"
print("\u2713 EDA plots created")


### How to interpret EDA for this task

Do not only show plots. Explain what they imply for modeling. For example, a strong positive skew in stars can affect topic prevalence, and strong city concentration can introduce regional language effects.

## 5. Text cleaning helpers

The cleaning here is intentionally simple. For Task 1, the goal is interpretable thematic comparison, not perfect NLP preprocessing.

In [ ]:
DOMAIN_STOPWORDS = {
    "yelp", "restaurant", "restaurants", "place", "places", "food", "foods", "menu",
    "order", "ordered", "ordering", "eat", "eating", "really", "just", "like", "good", "didn", "don", "ve", "ll"
}

def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"http\S+", " ", s)
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    s = re.sub(r"\b[a-z]\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

review_df["clean_text"] = review_df["text"].fillna("").map(clean_text)
comparison_df = review_df[review_df["rating_group"].isin(["high", "low"])].copy()
print("Comparison rows:", len(comparison_df))
display(comparison_df["rating_group"].value_counts())


### Downstream handoff: cleaned thematic review base

Task 1 generates the cleaned thematic review base that underpins Tasks 2–6. The exports below retain the selected restaurant/cuisine review subset, applying light cleaning and adding business metadata. This allows later notebooks to map categories, mine dish phrases, rank restaurants, analyze temporal trends, and build prediction features without redoing the filtering from Task 1.

In [ ]:
# Standard downstream handoff exports for Tasks 2-6.
# These files use stable, task-agnostic names in addition to the instructor-specific artifacts above.
handoff_review_subset = review_df.copy()
if "cleaned_text" not in handoff_review_subset.columns:
    handoff_review_subset["cleaned_text"] = handoff_review_subset.get("clean_text", "")

rename_for_handoff = {
    "name": "business_name",
    "categories_text": "categories",
}
handoff_review_subset = handoff_review_subset.rename(columns={k: v for k, v in rename_for_handoff.items() if k in handoff_review_subset.columns})

review_subset_columns = [
    "review_id", "business_id", "user_id", "date", "stars", "text", "cleaned_text",
    "business_name", "categories", "city", "state",
]
review_subset_columns = [c for c in review_subset_columns if c in handoff_review_subset.columns]
review_subset_export = handoff_review_subset[review_subset_columns].copy()
review_subset_export.to_csv(ARTIFACT_DIR / "review_subset.csv", index=False)

sample_n = min(2000, len(review_subset_export))
cleaned_reviews_sample = review_subset_export.sample(n=sample_n, random_state=RANDOM_SEED) if sample_n else review_subset_export.copy()
cleaned_reviews_sample.to_csv(ARTIFACT_DIR / "cleaned_reviews_sample.csv", index=False)

print("Saved downstream review subset:", ARTIFACT_DIR / "review_subset.csv")
print("Saved cleaned review sample:", ARTIFACT_DIR / "cleaned_reviews_sample.csv")
display(review_subset_export.head())

**Observation:** (write 1–2 sentences): What does this result tell you about the data or method?

The handoff export retains all **43,371** Mexican-restaurant reviews with both raw `text` and `cleaned_text` (every row has non-empty cleaned text) plus business/location metadata, giving downstream Tasks 2–6 a single, reproducible review base instead of re-filtering from scratch. Because the subset stays skewed toward high-rated reviews, later high-vs-low comparisons should be made within rating groups rather than on the pooled corpus.

### Text cleaning guidance

Use simple, explainable cleaning that removes obvious noise while preserving food, service, and experience terms. Keep preprocessing stable so n-gram and topic-model outputs are comparable.

## Data Mining Concept: Three Paradigms for Discovering Text Themes — Frequency, Probability, and Geometry

**The text-mining question:**

Given thousands of restaurant reviews, what are the **major themes** or **topics** that reviewers discuss? Are high-rated and low-rated reviews talking about different things?

This is a **theme discovery** problem, not a search problem. Because the themes are not known in advance, the goal is to let the data reveal what matters most to reviewers—such as food quality, service speed, ambiance, price, and specific dishes.

**The following three methods represent three fundamentally different paradigms for discovering themes:**

### Approach 1: N-gram frequency (Section 6) — The Frequency Paradigm

**Core idea**: The most important themes are the **phrases that appear most often**. Count 1-grams (words), 2-grams ("great service"), and 3-grams ("highly recommend this"), filter by frequency and stopwords, and treat the top-N n-grams as discovered themes.

**What this paradigm assumes:**
- Importance = frequency. If "good food" appears 500 times and "authentic flavors" appears 50 times, "good food" is 10× more important.
- Themes should be understood as **lexical surface patterns**, not latent concepts. Thus, “great service” and “excellent staff” are best treated as distinct themes, rather than grouped together as variants of a single *service quality* concept.
  - Lexical surface patterns are the exact words and phrases that appear in the text — for example, the literal strings “great service” or “excellent staff.” When we treat themes this way, each distinct wording counts as its own theme because we are only looking at what people actually wrote on the surface.
  - Latent concepts are underlying ideas or meanings that may be expressed with many different wordings — for example, the broader idea of service quality, which could show up as “great service,” “excellent staff,” “attentive waiters,” or “slow, rude servers.” When we focus on latent concepts, we try to group different phrases together because they point to the same underlying idea.
- No model, no parameters, no training — just counting.

**Strengths**: Transparent (you see exactly what reviewers wrote), fast, reproducible, vocabulary-preserving (you keep the original words).

**Weaknesses**:
- No abstraction — "delicious pizza", "amazing pizza", and "great pizza" are three separate n-grams, not one *pizza quality* theme.
- Dominated by common phrases — "the food was" might rank high but tells you nothing thematic.
- Cannot discover **latent topics** — there's no "atmosphere" theme unless reviewers literally use that word.

### Approach 2: Classical topic modeling with LDA (Section 7) — The Probability Paradigm

**Core idea**: Reviews are **mixtures of latent topics**, and topics are **probability distributions over words**. Use a generative probabilistic model (Latent Dirichlet Allocation) to infer K topics by finding word co-occurrence patterns across documents.
- In this context, a latent topic is an underlying theme or subject that is not directly labeled or observed in the data, but is inferred from patterns of word usage across many reviews. More concretely:
  - It’s “latent” because you don’t see it explicitly in the text (no tag saying “this is a food topic”); you discover it indirectly.
  - It’s a “topic” because it corresponds to a coherent theme, represented mathematically as a probability distribution over words (e.g., a topic about food might put high probability on words like delicious, meal, restaurant, taste).
- So a latent topic is a hidden thematic pattern in the corpus that the model uncovers by analyzing which words tend to appear together in documents (reviews).

**What this paradigm assumes:**
- Each review is a **bag of words** generated by sampling from K topics (e.g., 20% *food quality* topic, 60% *service* topic, 20% *price* topic).
- Words that **co-occur frequently across documents** belong to the same topic. If "spicy", "flavorful", "authentic" appear together in many reviews, they likely form a *cuisine-quality* topic.
- Topics are **interpretable distributions**: Topic 3 might assign high probability to {service, staff, friendly, attentive, slow} — you label it "service quality" by inspecting top terms.

**Strengths**:
- Discovers **latent abstract themes** not visible in raw n-gram counts.
- Handles **vocabulary variation**: "delicious", "tasty", "amazing" can all load onto the same *food-quality* topic.
- Produces topic proportions per document — you can measure "how much does this review discuss food vs. service?"

**Weaknesses**:
- **Requires choosing K** (number of topics) — too few topics = mushy themes, too many = incoherent splits.
- **Bag-of-words assumption**: ignores word order, so "not good" and "very good" contribute the same terms to topic distributions.
- **Interpretability is manual**: LDA gives you term probabilities, but *you* must decide what "Topic 7" means by reading its top words.
- **Nondeterministic**: different random seeds produce different topics (LDA uses sampling-based inference).

### Approach 3: Modern embedding-based clustering (Section 8, BERTopic) — The Geometry Paradigm

**Core idea**: Embed documents into a **high-dimensional semantic space** using neural sentence encoders (e.g., SBERT), reduce dimensions to 2D/3D using UMAP, find **dense clusters** using HDBSCAN, and label each cluster with its most distinctive terms (c-TF-IDF).

**What this paradigm assumes:**
- Semantically similar reviews are **geometrically close** in embedding space. Reviews about "slow service" and "long wait times" are near each other even if they use different words.
- Themes are **density islands** in this space, not probability distributions. Clustering algorithms find regions where reviews are tightly packed.
- Contextualized embeddings (BERT-based) capture **semantic meaning** — "not great" and "disappointing" are close in embedding space even though they share no words.

**Strengths**:
- **No need to choose K** upfront (HDBSCAN discovers the number of clusters from data density).
- **Semantic similarity**: Can group "terrible service" and "rude staff" into one theme even without lexical overlap.
- **Captures context**: "The sauce was fire" (positive) vs. "There was a fire in the kitchen" (negative) — embeddings distinguish meaning from word identity.

**Weaknesses**:
- **Black-box embeddings**: You don't know *why* two reviews are close in embedding space — the neural model's internal logic is opaque.
- **Computationally expensive**: Encoding thousands of reviews with SBERT is slower than TF-IDF.
- **Cluster quality varies**: HDBSCAN may find 3 coherent clusters and 15 tiny noise clusters — manual filtering is often needed.
- **Less interpretable term weights**: c-TF-IDF (a variant of TF-IDF computed per class/cluster rather than per individual document, to find distinctive terms for each cluster.) gives you distinctive terms per cluster, but the connection between embeddings and those terms is indirect.

**Key Insights 💡**

These three paradigms represent the **evolution of unsupervised text mining**:
1. **Terms Frequency** (1990s IR): Count what you see. Transparent but shallow.
2. **Probability** (2000s topic modeling): Model what you *don't* see (latent topics). Interpretable but requires assumptions (bag-of-words, fixed K).
3. **Geometry** (2010s-2020s neural embeddings): Embed documents into learned semantic space, cluster by density. Powerful but opaque.

**None is universally better.** The right method depends on:
- **Interpretability needs**: If you must *explain* themes to a stakeholder (e.g., Yelp product team), n-grams or LDA are easier to justify than BERT embeddings.
- **Vocabulary diversity**: If reviews use highly varied language ("delish", "yummy", "scrumptious"), embeddings help. If reviews are formulaic ("good food, good service"), n-grams suffice.
- **Computational budget**: Typically, N-grams run in seconds. LDA takes minutes. BERTopic can take 10-30 minutes for large corpora.

**What should you compare in your reports:**

1. **Theme coherence**: Do LDA topics and BERTopic clusters *make sense*? Can you label them with short human-readable descriptions?
2. **Coverage**: Do n-grams capture themes that LDA/BERTopic miss (e.g., specific dish names)? Do LDA/BERTopic find abstract themes ("ambiance") that n-grams miss?
3. **High-vs-low rating differences**: Do the three methods *agree* on what distinguishes 5-star reviews from 1-star reviews? If not, which method's answer is more useful?
4. **Stability**: If you re-run LDA with a different seed or BERTopic with slightly different parameters, do the themes stay consistent?

**The comparisons below** show how these three paradigms reveal different — but complementary — views of what Yelp reviewers care about.

## 6. Approach 1 — N-gram frequency analysis

This section uses simple count-based n-gram analysis as the first lens. It is intentionally transparent and easy to interpret.

You should compare what high-rated and low-rated reviews emphasize. This often reveals obvious but important signals: praise for dishes/service/ambience versus complaints about wait time, price, bland food, or poor service.

## About N-gram Frequency Analysis

N-gram frequency analysis is a transparent lexical method that counts how often contiguous word sequences (for example, unigrams, bigrams, and trigrams) appear in review text. In Task 1, it functions as the first interpretable baseline for answering the question, "What are people repeatedly talking about?" Because the method is count-based and easy to audit, you can quickly detect recurring dish terms, service phrases, and sentiment-laden expressions before moving to latent or embedding-driven methods. The same framework also supports side-by-side comparisons across rating groups, making it useful for high-vs-low diagnostic contrasts in Yelp reviews.

In the capstone pipeline, this method grounds interpretation and keeps model claims honest: if a topic model surfaces a theme that has weak lexical support, you can test whether the pattern appears in observable phrase frequencies. N-grams are therefore a strong bridge between exploratory text mining and formal topic discovery.

**Key Resources:**
- [Wikipedia: N-gram](https://en.wikipedia.org/wiki/N-gram) - Conceptual background on contiguous token sequences and language modeling foundations.
- [scikit-learn CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) - Practical API for building n-gram count representations.
- [Manning et al., Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/) - Classical text mining and term statistics reference used in many graduate courses.

Why use this method?
- Highly interpretable evidence for report-ready comparisons.
- Fast baseline for validating whether later methods capture real language patterns.
- Useful for identifying candidate phrase vocabularies feeding downstream tasks.

In [ ]:
# TODO (Category B — Classic data mining): Implement n-gram frequency analysis with CountVectorizer.
# Estimated time: ~1.5 h
#
# Expected inputs:
#   review_df       : full review subset, column "clean_text"
#   comparison_df   : high/low rating review subset, columns "clean_text", "rating_group"
#   DOMAIN_STOPWORDS: set of generic domain words to exclude (defined above)
#   NGRAM_SAMPLE_N  : max sample size (config constant)
#   RANDOM_SEED     : int
#
# Expected outputs (all required):
#   overall_ngrams  : DataFrame [ngram, count]  — top unigrams+bigrams across all reviews
#   group_ngrams    : DataFrame [rating_group, ngram, count] — per rating group
#   ARTIFACT_DIR / "task1_overall_top_ngrams.csv"
#   ARTIFACT_DIR / "task1_high_low_top_ngrams.csv"
#
# ── Implement top_ngrams() ──────────────────────────────────────────────────
# top_ngrams(texts, ngram_range, top_k, min_df, max_df) → DataFrame[ngram, count]
#
# Steps:
#  1. Convert texts to a pd.Series, drop NaN, cast to str.  Return empty DF if len==0.
#  2. Fit CountVectorizer(stop_words="english", ngram_range=ngram_range, min_df=min_df,
#       max_df=max_df, token_pattern=r"(?u)\b\w[\w']{1,}\b").
#  3. X = vec.fit_transform(texts)  [sparse matrix, shape (n_docs, n_vocab)]
#  4. counts = np.asarray(X.sum(axis=0)).ravel()   ← per-term document sums
#     terms  = vec.get_feature_names_out()
#  5. Build DataFrame {"ngram": terms, "count": counts}.
#  6. Filter: drop rows where ANY token in the ngram string is in DOMAIN_STOPWORDS.
#     Hint: ~freq["ngram"].apply(lambda g: any(tok in DOMAIN_STOPWORDS for tok in g.split()))
#  7. Return top_k rows sorted by count descending, reset index.
def top_ngrams(texts, ngram_range=(1, 2), top_k=25, min_df=5, max_df=0.70):
    texts = pd.Series(texts).dropna().astype(str)
    if len(texts) == 0:
        return pd.DataFrame(columns=["ngram", "count"])
    # TODO: implement CountVectorizer fit, frequency counting, DOMAIN_STOPWORDS filtering, and top-k selection
    # my code
    # create an instance of CountVectorizer to convert text into a matrix of token counts
    instance_of_CountVectorizer=CountVectorizer(
        stop_words="english",
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df,
        token_pattern=r"(?u)\b\w[\w']{1,}\b",
    )
    # fit the instance of CountVectorizer to the texts
    word_frequency_matrix=instance_of_CountVectorizer.fit_transform(texts)
    # get the total number of occurrences of each n-gram
    ngram_total_counts=np.asarray(word_frequency_matrix.sum(axis=0)).ravel()
    # get the names of the n-grams
    ngram_names=instance_of_CountVectorizer.get_feature_names_out()
    # build the dataframe for the n-gram frequency table
    ngram_frequency_table=pd.DataFrame({
        "ngram": ngram_names,
        "count": ngram_total_counts,
    })
    # create an internal function to check if the n-gram contains domain stopwords
    def contains_domain_stopword(ngram_text):
        # split the input n-gram into tokens
        tokens_in_ngram=ngram_text.split()
        # check if any token in the n-gram is in the domain stopwords
        for single_token in tokens_in_ngram:
            if single_token in DOMAIN_STOPWORDS:
                return True
        return False    
    # filter out the n-grams that contain domain stopwords
    ngram_frequency_table=ngram_frequency_table[
        ~ngram_frequency_table["ngram"].apply(contains_domain_stopword)
    ]
    # sort them by count descending and get the top k
    top_ngram_table=(
        ngram_frequency_table
        .sort_values("count", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )
    return top_ngram_table

# ── Compute overall_ngrams ──────────────────────────────────────────────────
# TODO: Sample review_df to at most NGRAM_SAMPLE_N rows (random_state=RANDOM_SEED).
#       Call top_ngrams with ngram_range=(1,2), top_k=30,
#       min_df=3 if DATA_SOURCE=="smoke_test" else 20.

# my code
# set the minimum document frequency
minimum_document_frequency=3 if DATA_SOURCE == "smoke_test" else 20
# sample the reviews to at most NGRAM_SAMPLE_N rows
number_of_reviews_to_sample=min(NGRAM_SAMPLE_N, len(review_df))
sample_for_ngrams=review_df.sample(
    n=number_of_reviews_to_sample,
    random_state=RANDOM_SEED,
)
# call top_ngrams to calculate the overall top n-grams
overall_ngrams=top_ngrams(
    texts=sample_for_ngrams["clean_text"],
    ngram_range=(1, 2),
    top_k=30,
    min_df=minimum_document_frequency,
    max_df=0.70,
)

# ── Compute group_ngrams per rating group ───────────────────────────────────
# TODO: Iterate over comparison_df.groupby("rating_group").
#       For each group, sample up to NGRAM_SAMPLE_N // 2 rows and call top_ngrams.
#       Add a "rating_group" column to each result.
#       Concatenate all group results into group_ngrams.

# my code
# set the maximum number of reviews to sample for each group
maximum_number_of_reviews_per_group=NGRAM_SAMPLE_N // 2   
# create a list to collect the results for the high and low rating groups
group_result_list=[]    
# iterate over the comparison_df by rating_group
for rating_group_name, group_reviews in comparison_df.groupby("rating_group"):
    # set the limit of reviews to sample for the current group
    number_to_sample=min(maximum_number_of_reviews_per_group, len(group_reviews))
    # sample the reviews for the current group
    sampled_group_reviews=group_reviews.sample(
        n=number_to_sample,
        random_state=RANDOM_SEED,
    )
    # call top_ngrams to calculate the top n-grams for the current group
    group_top_ngrams=top_ngrams(
        texts=sampled_group_reviews["clean_text"],
        ngram_range=(1, 2),
        top_k=30,
        min_df=minimum_document_frequency,
        max_df=0.70,
    )
    # add the rating group name to the group top n-grams
    group_top_ngrams["rating_group"]=rating_group_name
    # append the group top n-grams to the group result list
    group_result_list.append(group_top_ngrams)
# concatenate all the group results into a single dataframe
group_ngrams=pd.concat(group_result_list, ignore_index=True)

# ── Display and save ────────────────────────────────────────────────────────
if overall_ngrams is not None:
    display(overall_ngrams.head(15))
if group_ngrams is not None:
    display(group_ngrams.groupby("rating_group").head(10))
    overall_ngrams.to_csv(ARTIFACT_DIR / "task1_overall_top_ngrams.csv", index=False)
    group_ngrams.to_csv(ARTIFACT_DIR / "task1_high_low_top_ngrams.csv", index=False)

# --- Validation (uncomment after implementing) ---
assert isinstance(overall_ngrams, pd.DataFrame) and not overall_ngrams.empty
assert set(overall_ngrams.columns) >= {"ngram", "count"}
assert isinstance(group_ngrams, pd.DataFrame) and "rating_group" in group_ngrams.columns
assert (ARTIFACT_DIR / "task1_overall_top_ngrams.csv").exists()
print(f"\u2713 N-gram analysis: {len(overall_ngrams)} overall terms, groups: {group_ngrams['rating_group'].unique()}")


In [ ]:
def plot_top_ngrams(tbl, title, filename):
    if tbl.empty:
        print("No data to plot for", title)
        return
    plot_tbl = tbl.head(20).sort_values("count")
    plt.figure(figsize=(8, 6))
    plt.barh(plot_tbl["ngram"], plot_tbl["count"])
    plt.title(title)
    plt.xlabel("Count")
    plt.ylabel("n-gram")
    save_current_figure(filename)
    plt.show()
plt.close()

plot_top_ngrams(overall_ngrams, f"Top n-grams — {TARGET_CATEGORY} reviews", "task1_overall_top_ngrams.png")

for group in ["high", "low"]:
    tbl = group_ngrams[group_ngrams["rating_group"] == group]
    plot_top_ngrams(tbl, f"Top n-grams — {group}-rated {TARGET_CATEGORY} reviews", f"task1_{group}_top_ngrams.png")


### Interpreting n-gram evidence

Look for both shared and differentiating phrases. Go beyond polarity words and explain which aspects differ across high and low reviews (taste, freshness, wait time, service quality, pricing, or ambience).

## 7. Approach 2 — Classical Topic Modeling With LDA

LDA is the classical approach to topic modeling. It offers a useful point of comparison with the simpler n-gram frequency counts, and with modern embedding-based topic clustering techniques.

In this notebook, we present two complementary views of LDA:

1. **Raw LDA baseline:** relies solely on standard English stopwords. This intentionally preserves a key limitation: Yelp topics are often dominated by broad review terms such as `food`, `good`, `place`, and `service`.
2. **Refined LDA view:** introduces a small, restaurant-review-specific stopword list applied exclusively during topic modeling. The objective is not to strip sentiment or service-related language from all analyses, but to illustrate that topic-modeling preprocessing must be tailored to the question under investigation.

Because both views share the same high-versus-low target-category comparison framework, you can more easily discuss how preprocessing changes the themes that appear in the data.

### LDA Helper Functions

The helper cell below is designed to be reusable. It is called at multiple points
in the notebook as different LDA configurations are explored. Separating this helper
from the rest of the code clarifies which cells implement general-purpose machinery
and which cells capture specific modeling decisions.


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# TODO (Category B — Classic data mining): Implement the LDA topic modeling pipeline.
# Estimated time: ~2 h
#
# Implement TWO functions:
#
#  A) fit_lda_model(df, text_col, n_topics, sample_n, min_df, max_df,
#                   max_features, stop_words, ngram_range)
#     Returns: (lda_model, vectorizer, sample_df, doc_topic_matrix)
#
#     Steps:
#      1. Filter df to rows with non-empty text_col; sample up to sample_n rows
#         (random_state=RANDOM_SEED).
#      2. Fit CountVectorizer(stop_words, min_df, max_df, max_features, ngram_range,
#         token_pattern=r"(?u)\b\w[\w']{1,}\b") on sample[text_col].
#      3. Fit LatentDirichletAllocation(n_components=n_topics, learning_method="batch",
#         max_iter=15 (real data) or 5 (smoke_test), random_state=RANDOM_SEED,
#         evaluate_every=-1) on the document-term matrix X.
#      4. doc_topics = lda.fit_transform(X)   [shape: (n_docs, n_topics)]
#      5. Return (lda, vec, sample.reset_index(drop=True), doc_topics).
#
#  B) lda_topics_table(lda, vectorizer, top_n=12)
#     Returns: long-form DataFrame with columns [topic, rank, term, weight]
#
#     Steps:
#      1. terms = np.array(vectorizer.get_feature_names_out())
#      2. For each topic i and its weight vector lda.components_[i]:
#           argsort descending → top_n indices
#           emit rows: {topic: i, rank: 1..top_n, term: terms[idx], weight: float(weight)}
#      3. Return pd.DataFrame(rows).
#
# Hint: lda.components_ has shape (n_topics, vocab_size).
#       Use weights.argsort()[::-1][:top_n] to get the top-n term indices.

def fit_lda_model(
    df,
    text_col="clean_text",
    n_topics=8,
    sample_n=12_000,
    min_df=5,
    max_df=0.60,
    max_features=8000,
    stop_words="english",
    ngram_range=(1, 1),
):
    sample = df.dropna(subset=[text_col]).copy()
    sample = sample[sample[text_col].str.len() > 0]
    sample = sample.sample(n=min(sample_n, len(sample)), random_state=RANDOM_SEED)
    # TODO: build CountVectorizer and fit_transform on sample[text_col] → X

    # my code
    # create an instance of CountVectorizer to convert text 
    # into a matrix of token counts
    instance_of_Count_Vectorizer=CountVectorizer(
        stop_words=stop_words,
        min_df=min_df,
        max_df=max_df,
        max_features=max_features,
        ngram_range=ngram_range,
        token_pattern=r"(?u)\b\w[\w']{1,}\b",
    )
    # fit and transform the text into a document-term matrix
    document_term_matrix=instance_of_Count_Vectorizer.fit_transform(sample[text_col])
    
    # TODO: build and fit LatentDirichletAllocation; compute doc_topics = lda.fit_transform(X)
    # my code
    # set the maximum number of iterations for LDA
    maximum_number_of_iterations=5 if DATA_SOURCE == "smoke_test" else 15
    # create and train the LDA model
    lda_model=LatentDirichletAllocation(
        n_components=n_topics,
        learning_method="batch",
        max_iter=maximum_number_of_iterations,
        random_state=RANDOM_SEED,
        evaluate_every=-1,
    )
    # fit the LDA model on the document-term matrix
    doc_topic_matrix=lda_model.fit_transform(document_term_matrix)


    # TODO: return (lda, vec, sample.reset_index(drop=True), doc_topics)   
    # my code
    return (
        lda_model,
        instance_of_Count_Vectorizer,
        sample.reset_index(drop=True),
        doc_topic_matrix,
    )


def lda_topics_table(lda, vectorizer, top_n=12):
    terms = np.array(vectorizer.get_feature_names_out())
    # TODO: for each topic in lda.components_, extract top_n terms by weight
    # TODO: build and return long-form DataFrame [topic, rank, term, weight]
    
    # my code
    # create a list to collect the results for each topic
    topic_term_rows=[]
    # iterate each topic in the LDA model
    for topic_id, topic_word_weights in enumerate(lda.components_):
        # sort the topic word weights in descending order and get the top top_n terms
        indexes_of_top_terms=topic_word_weights.argsort()[::-1][:top_n]
        # iterate all the top terms
        for rank, word_index in enumerate(indexes_of_top_terms, start=1):
            topic_term_rows.append({
                "topic": topic_id,
                "rank": rank,
                "term": terms[word_index],
                "weight": float(topic_word_weights[word_index]),
            })
    # assemble the results into a DataFrame and return it
    topic_terms_table=pd.DataFrame(topic_term_rows)
    return topic_terms_table

In [ ]:
lda_min_df = 2 if DATA_SOURCE == "smoke_test" else 15
lda_model, lda_vec, lda_sample, lda_doc_topics = fit_lda_model(
    comparison_df,
    n_topics=N_LDA_TOPICS,
    sample_n=LDA_SAMPLE_N,
    min_df=lda_min_df,
    max_df=0.70,
    max_features=6000,
)

lda_topics = lda_topics_table(lda_model, lda_vec, top_n=12)
display(lda_topics.groupby("topic").head(8))
lda_topics.to_csv(ARTIFACT_DIR / "task1_lda_topics.csv", index=False)

def plot_lda_topic_bars(
    topic_table,
    topics_to_plot=None,
    top_n=10,
    filename_prefix="task1_lda_topic",
    title_prefix="LDA Topic",
):
    if topic_table is None or topic_table.empty:
        print(f"Skipping {filename_prefix}: no topic terms available.")
        return
    if topics_to_plot is None:
        topics_to_plot = sorted(topic_table["topic"].unique())[:min(8, topic_table["topic"].nunique())]
    for topic in topics_to_plot:
        tbl = topic_table[topic_table["topic"] == topic].head(top_n).sort_values("weight")
        if tbl.empty:
            continue
        plt.figure(figsize=(7, 4))
        plt.barh(tbl["term"], tbl["weight"])
        plt.title(f"{title_prefix} {topic}: top terms")
        plt.xlabel("LDA term weight")
        save_current_figure(f"{filename_prefix}_{topic}.png")
        plt.show()
    plt.close()


plot_lda_topic_bars(lda_topics, top_n=10)

### Topic Prevalence Helper

This helper ties your unsupervised topics directly to the assignment question: which themes are more prevalent in high-rated versus low-rated reviews? At this stage, topic modeling turns into concrete, reportable evidence rather than just a list of words.

In [ ]:
# TODO (Category C — Method design): Implement topic prevalence comparison by rating group.
# Estimated time: ~1 h
#
# topic_prevalence_by_rating(doc_topics, sample_df, prefix, title, filename)
#
# Expected input:
#   doc_topics : 2D numpy array, shape (n_docs, n_topics) — output of lda.fit_transform(X)
#   sample_df  : DataFrame aligned with doc_topics rows; must have "rating_group" column
#   prefix     : string prefix for saved CSV filename  (e.g. "task1_refined_lda")
#   title      : string title for the bar chart
#   filename   : filename for the saved PNG
#
# Expected output:
#   avg        : DataFrame — average topic weight per rating group
#                columns: topic, topic_num, high, low (or whichever groups are present)
#   Saved CSV  : ARTIFACT_DIR / f"{prefix}_topic_prevalence_by_rating.csv"
#   Saved plot : ARTIFACT_DIR / filename   (grouped bar chart: topic on x-axis, high vs low bars)
#   Returns    : avg
#
# Design choices to make:
#  - How do you go from doc_topics (per-document weights) to per-group averages?
#    Hint: pd.DataFrame(doc_topics, columns=[f"topic_{i}" for i in range(...)]),
#          then attach "rating_group" and use .groupby("rating_group").mean().T
#  - How do you build the side-by-side grouped bar chart?
#    Hint: use np.arange(len(plot_tbl)) for x positions, plt.bar with width offsets.
#  - Remember to call save_current_figure(filename) and plt.close().
def topic_prevalence_by_rating(doc_topics, sample_df, prefix, title, filename):
    # TODO: build a DataFrame of per-document topic weights with a rating_group column
    # TODO: compute average topic weight per rating group (.groupby().mean())
    # TODO: reshape to have topics as rows and rating groups as columns
    # TODO: create a grouped bar chart (high vs low) and save it
    # TODO: save the avg DataFrame as a CSV and return it
    # my code
    # 1. build a DataFrame of per-document topic weights with a rating_group column
    # get the number of topics, which is used for the column names
    number_of_topics=doc_topics.shape[1]
    # create the column names for the topics
    topic_column_names=[f"topic_{i}" for i in range(number_of_topics)]
    # build the DataFrame
    doc_topic_table=pd.DataFrame(doc_topics, columns=topic_column_names)
    # add a new column for the rating group
    doc_topic_table["rating_group"]=sample_df["rating_group"].values

    # 2. compute the average topic weight per rating group
    average_weight_by_group=(
        doc_topic_table
        .groupby("rating_group")[topic_column_names]
        .mean()
    )

    # 3. reshape the DataFrame to have topics as rows and rating groups as columns
    # transpose the DataFrame and reset the index
    average_prevalence_table=average_weight_by_group.T.reset_index()
    # rename the index column to topic
    average_prevalence_table=average_prevalence_table.rename(columns={"index": "topic"})
    # extract the topic number and convert it to an integer
    average_prevalence_table["topic_num"]=(
        average_prevalence_table["topic"]
        .str.replace("topic_", "", regex=False)
        .astype(int)
    )
    # rearrange the column order
    rating_group_columns=[c for c in average_weight_by_group.index if c in average_prevalence_table.columns]
    average_prevalence_table=average_prevalence_table[
        ["topic", "topic_num"] + rating_group_columns
    ]
    # 4. draw the grouped bar chart
    topic_labels=[f"Topic {i}" for i in average_prevalence_table["topic_num"]]
    x_positions=np.arange(len(average_prevalence_table))
    bar_width=0.35
    plt.figure(figsize=(10, 5))
    if "high" in average_prevalence_table.columns:
        plt.bar(
            x_positions - bar_width / 2,
            average_prevalence_table["high"],
            width=bar_width,
            label="high",
        )
    if "low" in average_prevalence_table.columns:
        plt.bar(
            x_positions + bar_width / 2,
            average_prevalence_table["low"],
            width=bar_width,
            label="low",
        )
    plt.xticks(x_positions, topic_labels, rotation=45, ha="right")
    plt.ylabel("Average topic weight")
    plt.title(title)
    plt.legend()
    save_current_figure(filename)
    plt.show()
    plt.close()
    # 5. save the average DataFrame as a CSV and return it
    csv_path=ARTIFACT_DIR / f"{prefix}_topic_prevalence_by_rating.csv"
    average_prevalence_table.to_csv(csv_path, index=False)
    return average_prevalence_table

### Main LDA Model: Topic-Aware Stopwords

LDA is sensitive to stopwords. The standard English stopword list alone produces topics dominated
by generic Yelp-review language — words like "food", "place", and "restaurant" appear everywhere.
To surface more meaningful themes, we fit LDA with a **domain-specific stopword list** that removes
common restaurant-review words while preserving cuisine and dish vocabulary.

This preprocessing choice is a core data-mining decision. The same reviews can produce very
different topics simply by changing which words are excluded before fitting. Your report should
document what was removed and why.

### Refined LDA Stopword List

Here, you have the chance to curate a stopword list specifically tailored for topic modeling. In the broader project, we keep words like `service`, `good`, and `great` because they are important for sentiment and thematic interpretation. However, for LDA we remove them so that latent topics are less dominated by generic review language and more reflective of substantive themes.

In [ ]:
# TODO (Category C — Method design): Curate domain-specific stopwords for restaurant review LDA.
# Estimated time: ~0.5 h
#
# Goal: extend sklearn's ENGLISH_STOP_WORDS with words that are so generic in restaurant reviews
# that they add noise to LDA topics without revealing meaningful themes.
#
# Think about:
#  - Very common restaurant-review words that appear in almost every topic regardless of theme
#    (e.g., "food", "restaurant", "good", "service", "staff", "place")
#  - Common filler/modal words unique to casual review writing
#    (e.g., "really", "definitely", "amazing", "awesome", "love")
#  - Words you already excluded in DOMAIN_STOPWORDS above — avoid double-counting
#
# Expected output:
#   RESTAURANT_REVIEW_TOPIC_STOPWORDS : sorted list of stopword strings
#
# Hint: start with set(ENGLISH_STOP_WORDS), add your domain words with .union({...}),
#       then wrap in sorted(...).
#
# After building the list, check its length to verify you added at least 15 domain words:
#   assert len(RESTAURANT_REVIEW_TOPIC_STOPWORDS) > len(ENGLISH_STOP_WORDS)

RESTAURANT_REVIEW_TOPIC_STOPWORDS = sorted(
    set(ENGLISH_STOP_WORDS).union({
        # TODO: add domain-specific stopwords for restaurant reviews
        # Include at least 15 words that are generic enough to appear in every topic

        "food", "restaurant", "restaurants", "place", "places",
        "service", "staff", "server", "wait", "waited", "waiting",
        "dining", "dinner", "lunch", "experience",
        "good", "great", "best", "nice", "love", "loved",
        "amazing", "awesome", "definitely", "really", "always",
        "atmosphere", "recommend", "recommended", "visit", "visited",
        "like", "just", "don", "ve", "didn", "got", "came", "said",
        "went", "one", "two", "three", "four", "five", "six","seven",
        "eight", "nine", "ten", "lot", "lots", "lot of", "a lot of",
        
    })
)

print(f"Stopword list length: {len(RESTAURANT_REVIEW_TOPIC_STOPWORDS)} "
      f"(vs. sklearn default: {len(list(ENGLISH_STOP_WORDS))})")
assert len(RESTAURANT_REVIEW_TOPIC_STOPWORDS) > len(list(ENGLISH_STOP_WORDS)), \
    "Add at least some domain words beyond sklearn defaults"
print("\u2713 Domain stopword list looks good")


### Fit the Refined LDA Model

The refined LDA model addresses the same high/low Mexican-review question, but with
a vocabulary specifically tuned for topic discovery. Preprocessing choices change
what evidence is available — a cleaner vocabulary often surfaces more coherent topics.


In [ ]:
# TODO (Category E — Guided sklearn): Fit the refined LDA model using your domain stopword list.
# Estimated time: ~0.5 h
#
# Expected input:
#   comparison_df               : DataFrame with "clean_text" column
#   N_LDA_TOPICS                : int (config constant)
#   LDA_SAMPLE_N                : int (config constant)
#   lda_min_df                  : int (set in the cell above)
#   RESTAURANT_REVIEW_TOPIC_STOPWORDS : your curated stopword list
#   REFINED_LDA_MAX_FEATURES    : int (config constant)
#   REFINED_LDA_NGRAM_RANGE     : tuple (config constant)
#   fit_lda_model()             : function you implemented above
#
# Expected output:
#   refined_lda_model, refined_lda_vec, refined_lda_sample, refined_lda_doc_topics
#   refined_lda_topics          : DataFrame from lda_topics_table()
#   Saved to                    : ARTIFACT_DIR / "task1_refined_lda_topics.csv"
#
# Call fit_lda_model(...) with:
#   n_topics=N_LDA_TOPICS, sample_n=LDA_SAMPLE_N, min_df=lda_min_df, max_df=0.70,
#   max_features=REFINED_LDA_MAX_FEATURES,
#   stop_words=RESTAURANT_REVIEW_TOPIC_STOPWORDS,
#   ngram_range=REFINED_LDA_NGRAM_RANGE
#
# Then call lda_topics_table() to extract the term table and display the top 8 terms per topic.

# TODO: call fit_lda_model and unpack the four return values
# my code
# call the fit_lda_model function with the given parameters
refined_lda_model, refined_lda_vec, refined_lda_sample, refined_lda_doc_topics=fit_lda_model(
    df=comparison_df,
    text_col="clean_text",
    n_topics=N_LDA_TOPICS,
    sample_n=LDA_SAMPLE_N,
    min_df=lda_min_df,
    max_df=0.70,
    max_features=REFINED_LDA_MAX_FEATURES,
    stop_words=RESTAURANT_REVIEW_TOPIC_STOPWORDS,
    ngram_range=REFINED_LDA_NGRAM_RANGE,
)

# TODO: call lda_topics_table() to get refined_lda_topics, display, and save
# my code
refined_lda_topics=lda_topics_table(
    lda=refined_lda_model,
    vectorizer=refined_lda_vec,
    top_n=12,
)
display(refined_lda_topics.groupby("topic").head(8))
refined_lda_topics.to_csv(ARTIFACT_DIR / "task1_refined_lda_topics.csv", index=False)

# --- Validation ---
assert refined_lda_model is not None, "fit_lda_model returned None — did you implement it?"
assert isinstance(refined_lda_topics, pd.DataFrame) and not refined_lda_topics.empty
assert (ARTIFACT_DIR / "task1_refined_lda_topics.csv").exists()
print(f"\u2713 Refined LDA: {refined_lda_topics['topic'].nunique()} topics extracted")


### Xiaofei
### Subset-specific refined LDA (high-only vs low-only)

In addition to a **unified** refined LDA on combined high+low reviews (shared topic space),
we fit **separate** refined LDA models on each rating subset. Topics are **not aligned**
across models (Topic 3 in high ≠ Topic 3 in low); the goal is to see which themes emerge
naturally within each group. One summary figure per subset is saved for the report.

In [ ]:
# Xiaofei
# --- Subset-specific refined LDA ---
high_df = comparison_df[comparison_df["rating_group"] == "high"].copy()
low_df = comparison_df[comparison_df["rating_group"] == "low"].copy()
print("high rows:", len(high_df), "| low rows:", len(low_df))

# 可与合并模型用相同 sample_n；不超过子集大小
high_sample_n = min(LDA_SAMPLE_N, len(high_df))
low_sample_n = min(LDA_SAMPLE_N, len(low_df))

lda_kwargs = dict(
    text_col="clean_text",
    n_topics=N_LDA_TOPICS,
    min_df=lda_min_df,
    max_df=0.70,
    max_features=REFINED_LDA_MAX_FEATURES,
    stop_words=RESTAURANT_REVIEW_TOPIC_STOPWORDS,
    ngram_range=REFINED_LDA_NGRAM_RANGE,
)

(
    high_lda_model, high_lda_vec, high_lda_sample, high_lda_doc_topics
) = fit_lda_model(df=high_df, sample_n=high_sample_n, **lda_kwargs)

(
    low_lda_model, low_lda_vec, low_lda_sample, low_lda_doc_topics
) = fit_lda_model(df=low_df, sample_n=low_sample_n, **lda_kwargs)

high_lda_topics = lda_topics_table(high_lda_model, high_lda_vec, top_n=12)
low_lda_topics = lda_topics_table(low_lda_model, low_lda_vec, top_n=12)

high_lda_topics.to_csv(ARTIFACT_DIR / "task1_refined_lda_topics_high_only.csv", index=False)
low_lda_topics.to_csv(ARTIFACT_DIR / "task1_refined_lda_topics_low_only.csv", index=False)

display(high_lda_topics.groupby("topic").head(5))
display(low_lda_topics.groupby("topic").head(5))
print("✓ Subset-specific refined LDA fitted")

In [ ]:
# xiaofei
def plot_lda_topics_grid(topic_table, title, filename, top_n=8, n_cols=4):
    """One PNG: grid of top-term bars for all topics."""
    topics = sorted(topic_table["topic"].unique())
    n_topics = len(topics)
    n_rows = (n_topics + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, topic in zip(axes, topics):
        tbl = (
            topic_table[topic_table["topic"] == topic]
            .head(top_n)
            .sort_values("weight")
        )
        ax.barh(tbl["term"], tbl["weight"])
        ax.set_title(f"Topic {topic}")
        ax.set_xlabel("weight")

    for ax in axes[n_topics:]:
        ax.axis("off")

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    save_current_figure(filename)
    plt.show()
    plt.close()


plot_lda_topics_grid(
    high_lda_topics,
    title="Refined LDA topics — high-rated reviews only",
    filename="task1_refined_lda_topics_grid_high_only.png",
)

plot_lda_topics_grid(
    low_lda_topics,
    title="Refined LDA topics — low-rated reviews only",
    filename="task1_refined_lda_topics_grid_low_only.png",
)

### Visualize and Compare Refined LDA Prevalence

Refined LDA will typically surface more concrete terms related to food, style, and service failures. The high/low prevalence view remains important: even a clear, well-defined topic is only useful if students can relate it to the specific comparison they were asked to make.

In [ ]:
plot_lda_topic_bars(
    refined_lda_topics,
    top_n=10,
    filename_prefix="task1_refined_lda_topic",
    title_prefix="Refined LDA Topic",
)

refined_avg_prevalence = topic_prevalence_by_rating(
    refined_lda_doc_topics,
    refined_lda_sample,
    prefix="task1_refined_lda",
    title="Refined LDA topic prevalence: high vs. low reviews",
    filename="task1_refined_lda_topic_prevalence_high_low.png",
)

### Xiaofei Refined LDA prevalence delta (high − low)

Grouped bars (previous figure) show absolute topic weights in each rating group.
This chart applies an **extra transformation** Δ = avg_weight(high) − avg_weight(low)
to make rating-associated differences easier to compare (Task 1.2).

In [ ]:
# xiaofei
# Extra transformation: topic prevalence delta (high-rated minus low-rated)
assert refined_avg_prevalence is not None and not refined_avg_prevalence.empty
assert {"high", "low"}.issubset(refined_avg_prevalence.columns), (
    "Need both 'high' and 'low' columns in refined_avg_prevalence"
)

delta_tbl = refined_avg_prevalence.copy()
delta_tbl["delta_high_minus_low"] = delta_tbl["high"] - delta_tbl["low"]
delta_tbl = delta_tbl.sort_values("delta_high_minus_low", ascending=True)

# Optional: human-readable y-labels from top LDA terms (uncomment if you want)
# top_term = (
#     refined_lda_topics.sort_values("weight", ascending=False)
#     .groupby("topic")["term"].first()
# )
# y_labels = [
#     f"Topic {int(r.topic_num)}: {top_term.get(r.topic, '')}"
#     for _, r in delta_tbl.iterrows()
# ]
y_labels = [f"Topic {int(r.topic_num)}" for _, r in delta_tbl.iterrows()]

colors = [
    "#2ca02c" if v >= 0 else "#d95f02"
    for v in delta_tbl["delta_high_minus_low"]
]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(y_labels, delta_tbl["delta_high_minus_low"], color=colors)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Avg topic weight: high-rated − low-rated")
ax.set_title("Topic prevalence delta — high vs low rated (Mexican, refined LDA)")
ax.invert_yaxis()  # largest positive delta at top (match common delta-chart layout)

# annotate bar ends (optional)
for i, v in enumerate(delta_tbl["delta_high_minus_low"]):
    ax.text(
        v + (0.01 if v >= 0 else -0.01),
        i,
        f"{v:+.3f}",
        va="center",
        ha="left" if v >= 0 else "right",
        fontsize=9,
    )

plt.tight_layout()
save_current_figure("task1_refined_lda_topic_prevalence_delta_high_low.png")
plt.show()
plt.close()

# save delta table for report / handoff
delta_tbl.to_csv(
    ARTIFACT_DIR / "task1_refined_lda_topic_prevalence_delta_high_low.csv",
    index=False,
)
print("saved delta prevalence chart and CSV")

## 8. Approach 3 — Modern Embedding/Topic-Clustering Analysis

## About Sentence-Transformer Embeddings

Sentence-transformer embeddings map review text into dense vector spaces where geometric distance approximates semantic similarity. In Task 1, they serve as the representational backbone for modern topic discovery, enabling the pipeline to group reviews by meaning rather than exact token overlap. This is particularly useful for Yelp data, where users describe similar dining experiences with highly variable vocabulary.

From a learning value perspective, sentence embeddings provide a clear contrast with classical bag-of-words features. Count-based representations are transparent but sparse and lexically rigid, whereas sentence embeddings are semantically expressive yet less directly interpretable. By examining downstream topics, prevalence tables, and their divergence from LDA and n-gram outputs, you can see how representation choice shapes which patterns become visible.

**Key Resources:**
- [Sentence-Transformers Documentation](https://www.sbert.net/) - Official documentation and model usage patterns.
- [Sentence-BERT Paper](https://aclanthology.org/D19-1410/) - Foundational architecture for semantic sentence embeddings.
- [Hugging Face Sentence-Transformers Models](https://huggingface.co/sentence-transformers) - Model catalog and deployment references.

## About BERTopic

BERTopic is a modern topic discovery framework that integrates semantic embeddings,
dimensionality reduction, density-based clustering, and class-based term weighting
to generate interpretable topic representations. In Task 1, it is used to contrast
embedding-driven theme discovery with classical count-based approaches such as
n-grams and LDA. This comparison is central to the capstone: you must assess what
each method discovers, what it misses, and the associated computational costs.

For review data, BERTopic's key advantage is its ability to cluster semantically
similar sentences even when users employ different wording. This reveals richer
patterns around dishes, service experiences, and ambiance than strictly lexical
methods. However, BERTopic can produce outlier assignments (topic -1) and relies
on heavier dependencies and model availability. Treat those diagnostics as model-health context and clearly label any fallbacks.

Within the Task 1 pipeline, BERTopic operates on the target-category high-vs-low
analysis, contrasting topic themes between high-rated and low-rated reviews.


In [ ]:
# TODO (Category A — Modern library): Implement the BERTopic pipeline with graceful fallback.
# Estimated time: ~3 h
#
# Implement TWO functions:
#
# ── A) run_bertopic_if_available(texts, metadata_df, analysis_label) ──────────
#
#  Returns: (topic_model, topics, info)
#    - topic_model : fitted BERTopic object
#    - topics      : list/array of topic assignments per document (-1 = outlier)
#    - info        : DataFrame from topic_model.get_topic_info()
#  Returns (None, None, None) if:
#    - DATA_SOURCE == "smoke_test"  (skip without error)
#    - BERTopic/sentence-transformers/umap/hdbscan cannot be imported
#    - BERTopic.fit_transform raises an exception
#
#  BERTopic pipeline to configure:
#    - Embedding model : os.environ.get("TASK1_SENTENCE_TRANSFORMER_MODEL", "all-MiniLM-L6-v2")
#    - UMAP            : n_neighbors=20, n_components=5, min_dist=0.0,
#                        metric="cosine", random_state=RANDOM_SEED
#    - HDBSCAN         : min_cluster_size=25, min_samples=5, metric="euclidean",
#                        cluster_selection_method="eom", prediction_data=True
#    - CountVectorizer : stop_words="english", ngram_range=(1,2), min_df=1, max_df=0.70
#    - BERTopic        : top_n_words=10, min_topic_size=25, verbose=True
#
# ── B) run_modern_fallback(texts, n_topics=8) ────────────────────────────────
#
#  TF-IDF + TruncatedSVD + KMeans fallback (used when BERTopic is unavailable).
#  Returns: (model_dict, clusters, info)
#    - model_dict : {"vectorizer": ..., "svd": ..., "kmeans": ...,
#                    "silhouette": float|nan, "topic_terms": DataFrame}
#    - clusters   : 1D array of cluster labels, length = len(texts)
#    - info        : DataFrame with columns ["Topic", "Count"]
#
#  Fallback pipeline:
#    1. TfidfVectorizer(stop_words="english", min_df=2 (smoke) / 10 (real),
#         max_df=0.70, max_features=8000, ngram_range=(1,2)) → X (sparse)
#    2. TruncatedSVD(n_components=min(50, X.shape[1]-1, X.shape[0]-1),
#         random_state=RANDOM_SEED) → Z (dense, shape n_docs × n_components)
#    3. KMeans(n_clusters=min(n_topics, len(texts)//20), random_state=RANDOM_SEED,
#         n_init=10).fit_predict(Z) → clusters
#    4. For each cluster, compute TF-IDF centroid over member docs, extract top-12 terms.
#    5. silhouette_score(Z, clusters) if >1 cluster; else np.nan.
#    6. Return model_dict, clusters, info DataFrame.
#
# Hint for centroid term extraction:
#   member_idx = np.where(clusters == c)[0]
#   centroid   = np.asarray(X[member_idx].mean(axis=0)).ravel()
#   top_idx    = centroid.argsort()[::-1][:12]

def run_bertopic_if_available(texts, metadata_df=None, analysis_label="modern"):
    if DATA_SOURCE == "smoke_test":
        print(f"Smoke-test mode: skipping BERTopic for {analysis_label} and using the local fallback.")
        return None, None, None
    try:
        from bertopic import BERTopic
        from sentence_transformers import SentenceTransformer  # noqa: F401
        from umap import UMAP
        from hdbscan import HDBSCAN
    except Exception as e:
        print(f"BERTopic stack unavailable for {analysis_label}; falling back. Import error: {e!r}")
        return None, None, None
    # TODO: configure and instantiate UMAP, HDBSCAN, CountVectorizer, and BERTopic
    # TODO: call topic_model.fit_transform(texts); wrap in try/except and return None-triple on failure
    # TODO: return (topic_model, topics, topic_model.get_topic_info())

    # load the embedding model
    embedding_model_name=os.environ.get(
        "TASK1_SENTENCE_TRANSFORMER_MODEL",
        "all-MiniLM-L6-v2",
    )
    embedding_model=SentenceTransformer(embedding_model_name)
    # configure the UMAP model according to the given parameters
    umap_model=UMAP(
        n_neighbors=20,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_SEED,
    )   
    # configure the HDBSCAN model according to the given parameters
    hdbscan_model=HDBSCAN(
        min_cluster_size=25,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    # configure the CountVectorizer model according to the given parameters
    vectorizer_model=CountVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.70,
    )
    # configure the BERTopic model according to the given parameters
    topic_model=BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        min_topic_size=25,
        verbose=True,
    )
    # try to fit the topic model
    try:
        topics, topic_probabilities=topic_model.fit_transform(texts)
        topic_info_table=topic_model.get_topic_info()
        return topic_model, topics, topic_info_table
    except Exception as e:
        print(f"BERTopic fit failed for {analysis_label}; falling back. Error: {e!r}")
        return None, None, None



def run_modern_fallback(texts, n_topics=8):
    # TODO: fit TfidfVectorizer on texts → X
    # TODO: fit TruncatedSVD on X → Z  (choose n_components safely: min(50, X.shape[1]-1, X.shape[0]-1))
    # TODO: fit KMeans on Z; k = min(n_topics, max(2, len(texts)//20))
    # TODO: for each cluster, compute the TF-IDF centroid and extract top-12 terms
    # TODO: compute silhouette_score (guard for edge cases)
    # TODO: return (model_dict, clusters, info)
    # clean the input texts
    cleaned_text_list=pd.Series(texts).dropna().astype(str)
    if len(cleaned_text_list)==0:
        empty_info_table=pd.DataFrame(columns=["Topic", "Count"])
        return {}, np.array([]), empty_info_table
    # set the minimum document frequency according to the given data source
    minimum_document_frequency=2 if DATA_SOURCE == "smoke_test" else 10
    # configure the TfidfVectorizer model according to the given parameters
    tfidf_vectorizer=TfidfVectorizer(
        stop_words="english",
        min_df=minimum_document_frequency,
        max_df=0.70,
        max_features=8000,
        ngram_range=(1, 2),
    )
    # fit the TfidfVectorizer model on the cleaned text list    
    tfidf_matrix=tfidf_vectorizer.fit_transform(cleaned_text_list)
    # configure the TruncatedSVD model according to the given parameters
    number_of_components=min(
        50,
        tfidf_matrix.shape[1]-1,
        tfidf_matrix.shape[0]-1,
    )
    # ensure the number of components is at least 2 
    number_of_components=max(2, number_of_components)
    # configure the TruncatedSVD model according to the given parameters
    svd_model=TruncatedSVD(
        n_components=number_of_components,
        random_state=RANDOM_SEED,
    )
    # reduce the dimensionality of the tfidf matrix 
    reduced_matrix=svd_model.fit_transform(tfidf_matrix)
    
    # configure the KMeans model according to the given parameters
    number_of_clusters=min(n_topics, max(2, len(cleaned_text_list)//20))
    kmeans_model=KMeans(
        n_clusters=number_of_clusters,
        random_state=RANDOM_SEED,
        n_init=10,
    )
    # fit the KMeans model on the reduced matrix
    cluster_labels=kmeans_model.fit_predict(reduced_matrix)
    
    # get the TF-IDF feature names
    feature_names=np.array(tfidf_vectorizer.get_feature_names_out())
    # create a list to store the topic term rows    
    topic_term_rows=[]
    # iterate all the unique cluster labels
    for cluster_id in sorted(np.unique(cluster_labels)):
        # get the member indices for the current cluster
        member_indices=np.where(cluster_labels==cluster_id)[0]
        # compute the TF-IDF centroid for the current cluster   
        cluster_centroid=np.asarray(tfidf_matrix[member_indices].mean(axis=0)).ravel()
        # get the top-12 terms for the current cluster
        top_term_indices=cluster_centroid.argsort()[::-1][:12]
        # iterate over the top-12 terms and keep them in the topic term rows
        for rank, term_index in enumerate(top_term_indices, start=1):
            topic_term_rows.append({
                "topic": int(cluster_id),
                "rank": rank,
                "term": feature_names[term_index],
                "weight": float(cluster_centroid[term_index]),
            })
    # convert the topic term rows to a pandas DataFrame 
    topic_terms_table=pd.DataFrame(topic_term_rows)
    
    # compute the silhouette score
    if len(np.unique(cluster_labels))>1:
        clustering_quality_score = float(silhouette_score(reduced_matrix, cluster_labels))
    else:
        clustering_quality_score = np.nan
    # get the cluster counts    
    cluster_counts=pd.Series(cluster_labels).value_counts().sort_index()
    # create the info table to store the cluster counts 
    info_table=pd.DataFrame({
        "Topic": cluster_counts.index.astype(int),
        "Count": cluster_counts.values,
    })
    # pack the model results and return them
    model_dict={
        "vectorizer":tfidf_vectorizer,
        "svd":svd_model,
        "kmeans":kmeans_model,
        "silhouette":clustering_quality_score,
        "topic_terms":topic_terms_table,
    }
    return model_dict, cluster_labels, info_table

### Build the Topic Sample

The target-category sample is used for both classical LDA and modern BERTopic comparison.

In [ ]:
def build_target_modern_topic_sample() -> pd.DataFrame:
    source = comparison_df.copy()
    source["modern_category"] = TARGET_CATEGORY
    source = source.dropna(subset=["clean_text"]).copy()
    source = source[source["clean_text"].str.len() > 0]
    return source.sample(
        n=min(MODERN_TARGET_SAMPLE_N, len(source)),
        random_state=RANDOM_SEED,
    ).reset_index(drop=True)


### Target-Category BERTopic

This part presents a modern alternative to traditional n-gram and LDA approaches. Applying
BERTopic (or a clearly labeled fallback) to the same target category lets you assess whether
modern embedding-based topics produce more coherent or actionable thematic structures than
classical methods.

In [ ]:
def build_target_modern_topic_sample() -> pd.DataFrame:
    source = comparison_df.copy()
    source["modern_category"] = TARGET_CATEGORY
    source = source.dropna(subset=["clean_text"]).copy()
    source = source[source["clean_text"].str.len() > 0]
    return source.sample(n=min(MODERN_TARGET_SAMPLE_N, len(source)), random_state=RANDOM_SEED).reset_index(drop=True)


def build_cross_cuisine_modern_topic_sample() -> pd.DataFrame:
    if DATA_SOURCE == "smoke_test":
        source = comparison_df.copy()
        source["modern_category"] = TARGET_CATEGORY
    else:
        modern_business_df = load_modern_topic_businesses(
            business_path,
            MODERN_TOPIC_CATEGORIES,
            require_restaurants=REQUIRE_RESTAURANTS,
        )
        print("Cross-cuisine modern-topic businesses by category:")
        display(modern_business_df["modern_category"].value_counts().rename_axis("modern_category").reset_index(name="businesses"))
        source = load_modern_topic_reviews(
            review_path,
            modern_business_df,
            per_category=MODERN_REVIEWS_PER_CATEGORY,
            chunksize=CHUNKSIZE,
        )
        source["clean_text"] = source["text"].fillna("").map(clean_text)

    source = source.dropna(subset=["clean_text"]).copy()
    source = source[source["clean_text"].str.len() > 0]
    return source.sample(n=min(MODERN_CROSS_CUISINE_SAMPLE_N, len(source)), random_state=RANDOM_SEED).reset_index(drop=True)


def fit_modern_analysis(sample_df: pd.DataFrame, analysis_scope: str):
    texts = sample_df["clean_text"].tolist()
    print(f"{analysis_scope} modern topic sample size:", len(texts))
    model_obj, topics, info = run_bertopic_if_available(texts, sample_df, analysis_label=analysis_scope)
    if model_obj is None:
        model_obj, topics, info = run_modern_fallback(texts, n_topics=N_MODERN_TOPICS_FALLBACK)
        method = "TF-IDF + SVD + KMeans fallback"
    else:
        method = "BERTopic"
    print(f"{analysis_scope} modern method used:", method)
    display(info.head(15))
    return model_obj, topics, info, method

modern_target_sample_df = build_target_modern_topic_sample()
modern_target_sample_summary = pd.crosstab(modern_target_sample_df["modern_category"], modern_target_sample_df["rating_group"])
display(modern_target_sample_summary)
modern_target_sample_summary.to_csv(ARTIFACT_DIR / "task1_modern_target_sample_by_rating.csv")

modern_target_model, modern_target_topics, modern_target_info, modern_target_method = fit_modern_analysis(
    modern_target_sample_df,
    analysis_scope="target_category",
)
modern_target_info.to_csv(ARTIFACT_DIR / "task1_modern_target_topic_info.csv", index=False)

### Extract and Plot Target-Category Modern Topic Terms

The topic-word table is the primary place to assess whether target-category BERTopic is identifying coherent themes. For standard interpretation, exclude topic `-1` from the topic-word table when it appears; treat it as a model-health diagnostic rather than a substantive theme.

In [ ]:
# TODO (Category D — Medium pipeline): Extract topic terms from BERTopic or fallback output.
# Estimated time: ~0.5 h
#
# Implement extract_modern_topic_terms(model_obj, method, info, max_topics=40)
#
# Expected input:
#   model_obj  : BERTopic instance  OR  fallback dict {"topic_terms": DataFrame, ...}
#   method     : string — "BERTopic" or any fallback label
#   info       : topic_info DataFrame (from get_topic_info() or fallback info)
#   max_topics : int, maximum number of topics to extract
#
# Expected output:
#   DataFrame with columns [topic, rank, term, weight, is_outlier]
#   Return empty DataFrame with those columns if info is None or empty.
#
# Two cases to handle:
#  Case 1 — method == "BERTopic":
#    Iterate over info["Topic"][:max_topics].
#    For each topic_id, call model_obj.get_topic(int(topic_id)) → list of (term, weight) pairs.
#    Build rows: {topic, rank, term, weight, is_outlier: topic_id == -1}
#
#  Case 2 — fallback:
#    Use model_obj["topic_terms"] directly; add is_outlier=False column.
#
# Also implement plot_modern_topic_bars(topic_terms, method, filename_prefix, title_prefix, top_n=10)
#   which plots a horizontal bar chart per non-outlier topic (up to 8 topics, top_n terms each).

def extract_modern_topic_terms(model_obj, method, info, max_topics=40):
    if info is None or len(info) == 0:
        return pd.DataFrame(columns=["topic", "rank", "term", "weight", "is_outlier"])
    # TODO: handle the BERTopic case (method == "BERTopic")

    # my code
    # case01: handle the BERTopic case
    # check if the method is BERTopic
    if method=="BERTopic":
        # create a list to store the topic term rows
        topic_term_rows=[]  
        # get the topic id list
        topic_id_list=info["Topic"].head(max_topics).tolist()
        # iterate over the topic id list
        for topic_id in topic_id_list:
            # convert the topic id to an integer    
            topic_id_int=int(topic_id)
            # get the topic terms
            term_weight_pairs=model_obj.get_topic(topic_id_int)
            # check if the topic terms is None
            if term_weight_pairs is None:
                continue
            # iterate over the topic terms
            for rank, (term, weight) in enumerate(term_weight_pairs, start=1):
                topic_term_rows.append({
                    "topic": topic_id_int,
                    "rank": rank,
                    "term": term,
                    "weight": float(weight),
                    "is_outlier": topic_id_int == -1,
                })
        # convert the topic term rows to a pandas DataFrame
        return pd.DataFrame(topic_term_rows)

    # TODO: handle the fallback case (model_obj is a dict with "topic_terms" key)
    # my code
    # case02: handle the fallback case
    # check if the current model is a result of fallback
    if isinstance(model_obj, dict) and "topic_terms" in model_obj:
        # create a copy of the topic terms to avoid modifying the original
        fallback_topic_table=model_obj["topic_terms"].copy()
        # add the is_outlier column and set it to False
        fallback_topic_table["is_outlier"]=False
        # return the topic terms table and keep the required columns
        return fallback_topic_table[["topic", "rank", "term", "weight", "is_outlier"]]
    # if the current model is not a result of fallback or a BERTopic instance, 
    # return an empty DataFrame
    return pd.DataFrame(columns=["topic", "rank", "term", "weight", "is_outlier"])

def plot_modern_topic_bars(topic_terms, method, filename_prefix, title_prefix, top_n=10):
    if topic_terms is None or topic_terms.empty:
        print(f"Skipping {filename_prefix}: no modern topic terms available.")
        return
    # TODO: for each non-outlier topic (up to 8), plot a horizontal bar chart of the top_n terms
    
    # my code
    # plot the non-outlier topics
    # create a copy of the topic terms to avoid modifying the original
    non_outlier_table=topic_terms[~topic_terms["is_outlier"]].copy()
    # filter out the outlier topic
    non_outlier_table=non_outlier_table[non_outlier_table["topic"] != -1]
    # if there is no non-outlier topic, skip the plot
    if non_outlier_table.empty:
        print(f"Skipping {filename_prefix}: only outlier topics available.")
        return
    # if there are more than 8 non-outlier topics, only plot the top 8
    topics_to_plot=sorted(non_outlier_table["topic"].unique())[:8]
    # iterate each topic to plot
    for topic_id in topics_to_plot:
        one_topic_table=(
            non_outlier_table[non_outlier_table["topic"] == topic_id]
            .sort_values("rank")
            .head(top_n)
            .sort_values("weight")  
        )
        # if the topic table is empty, skip the plot
        if one_topic_table.empty:
            continue
        # plot the topic terms
        plt.figure(figsize=(7, 4))
        plt.barh(one_topic_table["term"], one_topic_table["weight"])
        plt.title(f"{title_prefix} {topic_id}: top terms ({method})")
        plt.xlabel("Modern topic term weight")
        # TODO: save each plot with save_current_figure(f"{filename_prefix}_{topic}.png")
        save_current_figure(f"{filename_prefix}_{topic_id}.png")
        plt.show()
        plt.close()

In [ ]:
modern_target_topic_terms = extract_modern_topic_terms(
    modern_target_model,
    modern_target_method,
    modern_target_info,
)
display(modern_target_topic_terms[modern_target_topic_terms["topic"] != -1].groupby("topic").head(8))
modern_target_topic_terms.to_csv(ARTIFACT_DIR / "task1_modern_target_topic_terms.csv", index=False)
modern_target_topic_terms.to_csv(ARTIFACT_DIR / "task1_modern_topic_terms.csv", index=False)
plot_modern_topic_bars(
    modern_target_topic_terms,
    modern_target_method,
    filename_prefix="task1_modern_target_topic",
    title_prefix="Target-category BERTopic Topic",
)

### Modern Prevalence and Diagnostic Helpers

These helpers turn modern topics into clear, interpretable evidence by reporting rating-group prevalence, category prevalence, topic size and prevalence. Diagnostics are built into the analysis itself rather than tacked on as cleanup afterward.

In [ ]:
def modern_prevalence_tables(topics, sample_df, file_prefix):
    prev = pd.DataFrame({
        "rating_group": sample_df["rating_group"].values,
        "modern_category": sample_df.get("modern_category", pd.Series([TARGET_CATEGORY] * len(sample_df))).values,
        "topic": topics,
    })
    rating_crosstab = pd.crosstab(prev["topic"], prev["rating_group"], normalize="columns")
    display(rating_crosstab)
    rating_crosstab.to_csv(ARTIFACT_DIR / f"{file_prefix}_topic_prevalence_by_rating.csv")

    category_crosstab = pd.crosstab(prev["topic"], prev["modern_category"], normalize="columns")
    display(category_crosstab)
    category_crosstab.to_csv(ARTIFACT_DIR / f"{file_prefix}_topic_prevalence_by_category.csv")
    return prev, rating_crosstab, category_crosstab


def plot_modern_rating_prevalence(crosstab, method, filename, title):
    if crosstab is None or crosstab.empty:
        print(f"Skipping {filename}: no prevalence data.")
        return
    plot_tbl = crosstab.copy()
    plot_tbl = plot_tbl.loc[[idx for idx in plot_tbl.index if idx != -1]] if -1 in plot_tbl.index else plot_tbl
    plot_tbl = plot_tbl.head(12)
    if plot_tbl.empty:
        print(f"Skipping {filename}: only outlier topic available.")
        return
    plt.figure(figsize=(9, 4))
    x = np.arange(len(plot_tbl))
    width = 0.35
    if "high" in plot_tbl.columns:
        plt.bar(x - width/2, plot_tbl["high"], width=width, label="high")
    if "low" in plot_tbl.columns:
        plt.bar(x + width/2, plot_tbl["low"], width=width, label="low")
    plt.xticks(x, plot_tbl.index.astype(str))
    plt.xlabel("Modern topic/cluster")
    plt.ylabel("Share within rating group")
    plt.title(f"{title} ({method})")
    plt.legend()
    save_current_figure(filename)
    plt.show()
    plt.close()


def build_modern_topic_diagnostics(analysis_scope, method, info, topics, topic_terms, sample_df):
    topic_series = pd.Series(topics, name="topic")
    sample_size = len(topic_series)
    outlier_count = int((topic_series == -1).sum())
    non_outlier_topics = sorted(t for t in topic_series.dropna().unique() if int(t) != -1)
    term_lookup = (
        topic_terms.sort_values(["topic", "rank"])
        .groupby("topic")["term"]
        .apply(lambda s: "; ".join(s.astype(str).head(10)))
        .to_dict()
        if isinstance(topic_terms, pd.DataFrame) and not topic_terms.empty
        else {}
    )
    rows = []
    diag_df = sample_df.reset_index(drop=True).copy()
    diag_df["topic"] = topic_series.values
    for topic, group in diag_df.groupby("topic", dropna=False):
        cat_counts = group["modern_category"].value_counts() if "modern_category" in group.columns else pd.Series(dtype=int)
        rating_counts = group["rating_group"].value_counts() if "rating_group" in group.columns else pd.Series(dtype=int)
        dominant_category = cat_counts.index[0] if not cat_counts.empty else ""
        dominant_rating = rating_counts.index[0] if not rating_counts.empty else ""
        rows.append({
            "analysis_scope": analysis_scope,
            "method": method,
            "sample_size": sample_size,
            "n_non_outlier_topics": len(non_outlier_topics),
            "outlier_count": outlier_count,
            "outlier_rate": outlier_count / sample_size if sample_size else np.nan,
            "topic": int(topic),
            "is_outlier": int(topic) == -1,
            "topic_size": len(group),
            "topic_share": len(group) / sample_size if sample_size else np.nan,
            "dominant_category": dominant_category,
            "dominant_category_count": int(cat_counts.iloc[0]) if not cat_counts.empty else 0,
            "dominant_category_share": float(cat_counts.iloc[0] / len(group)) if not cat_counts.empty else np.nan,
            "dominant_rating_group": dominant_rating,
            "dominant_rating_group_count": int(rating_counts.iloc[0]) if not rating_counts.empty else 0,
            "dominant_rating_group_share": float(rating_counts.iloc[0] / len(group)) if not rating_counts.empty else np.nan,
            "top_terms": term_lookup.get(int(topic), ""),
        })
    return pd.DataFrame(rows).sort_values(["analysis_scope", "is_outlier", "topic"], ascending=[True, True, True])

### Target-Category Modern Prevalence

In the target-category analysis, the central comparison is between high and low review scores. A topic is most useful when its characteristic terms and its prevalence across reviews together support a coherent, consistent interpretation.

In [ ]:
# TODO (Category E — Guided): Call modern_prevalence_tables and save/plot results.
# Estimated time: ~0.5 h
#
# Expected input:
#   modern_target_topics     : array/list of topic assignments (from BERTopic or fallback run)
#   modern_target_sample_df  : DataFrame used for the modern topic run (with "rating_group" column)
#   modern_target_method     : string label, e.g. "BERTopic" or fallback name
#
# Expected output (three variables + saved artifacts):
#   modern_target_prev                 : per-document DataFrame
#   modern_target_crosstab             : topic × rating_group prevalence table
#   modern_target_category_crosstab    : topic × category prevalence table (if applicable)
#   Saved: ARTIFACT_DIR / "task1_modern_topic_prevalence_by_rating.csv"
#   Saved: ARTIFACT_DIR / "task1_modern_topic_prevalence_high_low.png"
#
# Call modern_prevalence_tables(topics, sample_df, file_prefix) with file_prefix="task1_modern_target".
# Then save modern_target_crosstab to the CSV above.
# Call plot_modern_rating_prevalence (defined below) twice:
#   once for "task1_modern_target_topic_prevalence_high_low.png"
#   once for "task1_modern_topic_prevalence_high_low.png"

# TODO: call modern_prevalence_tables and unpack three return values

# my code
# call modern_prevalence_tables and unpack three return values
modern_target_prev, modern_target_crosstab, modern_target_category_crosstab=modern_prevalence_tables(
    topics=modern_target_topics,
    sample_df=modern_target_sample_df,
    file_prefix="task1_modern_target",
)

# save the prevalence table to a CSV file
modern_target_crosstab.to_csv(
    ARTIFACT_DIR / "task1_modern_topic_prevalence_by_rating.csv"
)
# call plot_modern_rating_prevalence to plot the high vs low prevalence
# draw the first plot 
plot_modern_rating_prevalence(
    crosstab=modern_target_crosstab,
    method=modern_target_method,
    filename="task1_modern_target_topic_prevalence_high_low.png",
    title="Target-category modern topic prevalence by rating group",
)
# draw the second plot
plot_modern_rating_prevalence(
    crosstab=modern_target_crosstab,
    method=modern_target_method,
    filename="task1_modern_topic_prevalence_high_low.png",
    title="Target-category modern topic prevalence by rating group",
)
# TODO: save modern_target_crosstab to CSV
# my code
# build the modern topic diagnostics    
modern_target_diagnostics=build_modern_topic_diagnostics(
    analysis_scope="target_category",
    method=modern_target_method,
    info=modern_target_info,
    topics=modern_target_topics,
    topic_terms=modern_target_topic_terms,
    sample_df=modern_target_sample_df,
)
# save the modern topic diagnostics to a CSV file
modern_target_diagnostics.to_csv(
    ARTIFACT_DIR / "task1_modern_topic_diagnostics.csv",
    index=False,
)
# TODO: call plot_modern_rating_prevalence for both output filenames
display(modern_target_diagnostics)





# --- Validation ---
assert modern_target_crosstab is not None, "modern_target_crosstab is None — did you call modern_prevalence_tables?"
assert (ARTIFACT_DIR / "task1_modern_topic_prevalence_by_rating.csv").exists()
print("\u2713 Modern prevalence tables computed")


### Xiaofei Cell01-8.Supplementary diagnostics for the modern topic model (added cells — no required code modified)

The prescribed BERTopic configuration produced only **2 non-outlier topics** on 3,000
target-category reviews (topic 0 ≈ 97.6%, topic 1 ≈ 2.4%, 0 outliers). The cells below are
*exploratory supplements* that test whether this is a coding artifact or a genuine property
of the data + the (intentionally conservative) HDBSCAN settings. They write only new
artifacts and never alter the required pipeline or its outputs.

In [ ]:
# Xiaofei Cell01-8.Supplementary diagnostics — embedding prep (defines SUPP_TEXTS / SUPP_EMB / emb_ok)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SUPP_TEXTS = modern_target_sample_df["clean_text"].tolist()
print("Supplementary diagnostics on", len(SUPP_TEXTS), "target-category docs")

emb_ok = True
try:
    from sentence_transformers import SentenceTransformer
    _emb_name = os.environ.get("TASK1_SENTENCE_TRANSFORMER_MODEL", "all-MiniLM-L6-v2")
    _embedder = SentenceTransformer(_emb_name)
    SUPP_EMB = np.asarray(
        _embedder.encode(SUPP_TEXTS, show_progress_bar=False, normalize_embeddings=True),
        dtype="float32",
    )
    print("Embeddings:", SUPP_EMB.shape)
except Exception as e:
    emb_ok = False
    print("Embedding stack unavailable; supplementary diagnostics skipped:", repr(e))

In [ ]:
# Xiaofei Cell01-8.Supplementary diagnostics for the modern topic model (added cells — no required code modified)
# === Supplementary cell (revised) — exposes the levers that ACTUALLY drive topic count ===
# Change 1: _supp_fit_bertopic now accepts n_neighbors + cluster_selection_method,
#           and returns a "usable_topics" count (non-outlier topics holding >=1% of docs)
#           so we can distinguish real structure from over-fragmentation.
# Change 2: the sweep grid varies selection method / n_neighbors, not just min_cluster_size.
def _supp_fit_bertopic(embeddings, texts, min_cluster_size, min_samples,
                       n_neighbors=20, cluster_selection_method="eom"):
    from bertopic import BERTopic
    from umap import UMAP
    from hdbscan import HDBSCAN
    from sklearn.feature_extraction.text import CountVectorizer
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=5, min_dist=0.0,
                      metric="cosine", random_state=RANDOM_SEED)
    hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples,
                            metric="euclidean",
                            cluster_selection_method=cluster_selection_method,
                            prediction_data=True)
    vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2),
                                       min_df=1, max_df=0.70)
    tm = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model,
                  vectorizer_model=vectorizer_model, top_n_words=10,
                  min_topic_size=min_cluster_size, verbose=False)
    topics, _ = tm.fit_transform(texts, embeddings=embeddings)  # reuse cached embeddings
    info = tm.get_topic_info()
    t = np.asarray(topics)
    sizes = info[info["Topic"] != -1]["Count"]
    n_topics = int((info["Topic"] != -1).sum())
    usable = int((sizes / len(t) >= 0.01).sum())  # topics with >=1% of docs
    largest_share = float(sizes.max() / len(t)) if len(sizes) else np.nan
    return n_topics, usable, float((t == -1).mean()), largest_share


if emb_ok:
    # (label, min_cluster_size, min_samples, n_neighbors, selection)
    sweep_grid = [
        ("prescribed  eom mcs25 nn20", 25, 5, 20, "eom"),   # reproduces the CSV (2 topics)
        ("eom  mcs25 nn10",            25, 5, 10, "eom"),    # smaller neighborhood, same selection
        ("leaf mcs25 nn20",            25, 5, 20, "leaf"),   # <-- strongest single lever
        ("leaf mcs15 nn15",            15, 5, 15, "leaf"),
        ("leaf mcs10 nn10",            10, 3, 10, "leaf"),
        ("eom  mcs5  nn10",             5, 1, 10, "eom"),    # over-fragmentation reference
    ]

    rows = []
    for label, mcs, ms, nn, sel in sweep_grid:
        try:
            n_t, usable, o_rate, share = _supp_fit_bertopic(
                SUPP_EMB, SUPP_TEXTS, mcs, ms,
                n_neighbors=nn, cluster_selection_method=sel,
            )
        except Exception as e:
            n_t, usable, o_rate, share = (np.nan, np.nan, np.nan, np.nan)
            print(f"[{label}] failed: {e!r}")
        rows.append({
            "config": label, "min_cluster_size": mcs, "min_samples": ms,
            "n_neighbors": nn, "selection": sel,
            "n_non_outlier_topics": n_t, "usable_topics_ge1pct": usable,
            "outlier_rate": o_rate, "largest_topic_share": share,
        })
        print(f"[{label}] topics={n_t} usable={usable} "
              f"outlier_rate={o_rate if isinstance(o_rate,float) else o_rate:.3f} "
              f"largest={share if isinstance(share,float) else share:.3f}")

    sweep_df = pd.DataFrame(rows)
    sweep_df.to_csv(ARTIFACT_DIR / "task1_modern_param_sensitivity.csv", index=False)
    display(sweep_df)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(sweep_df["config"], sweep_df["usable_topics_ge1pct"], color="#55A868")
    ax.set_ylabel("usable topics (≥1% of docs)")
    ax.set_title("Topic count is driven by selection method & n_neighbors,\nnot by min_cluster_size alone")
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "task1_modern_param_sensitivity.png", dpi=150)
    plt.show()

In [ ]:
# Xiaofei Cell01-8.Supplementary diagnostics — topic coherence (c_v) to harden the sensitivity findings
# Computes c_v coherence for LDA vs BERTopic prescribed (eom) vs BERTopic leaf,
# so we can argue the leaf sub-topics are genuinely more coherent, not just more numerous.
if emb_ok:
    import pandas as pd
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel

    _texts_tok = [str(t).split() for t in SUPP_TEXTS]
    _dictionary = Dictionary(_texts_tok)

    def _normalize_topic_words(topic_word_lists, top_n=10):
        norm = []
        for tw in topic_word_lists:
            toks = []
            for term in tw[:top_n]:
                for piece in str(term).split():
                    if piece in _dictionary.token2id and piece not in toks:
                        toks.append(piece)
            if len(toks) >= 2:
                norm.append(toks)
        return norm

    def _cv(topic_word_lists):
        topics = _normalize_topic_words(topic_word_lists)
        if len(topics) < 1:
            return float("nan")
        cm = CoherenceModel(topics=topics, texts=_texts_tok,
                            dictionary=_dictionary, coherence="c_v")
        return float(cm.get_coherence())

    def _bertopic_topic_words(mcs, ms, nn, sel, top_n=10):
        from bertopic import BERTopic
        from umap import UMAP
        from hdbscan import HDBSCAN
        from sklearn.feature_extraction.text import CountVectorizer
        umap_model = UMAP(n_neighbors=nn, n_components=5, min_dist=0.0,
                          metric="cosine", random_state=RANDOM_SEED)
        hdbscan_model = HDBSCAN(min_cluster_size=mcs, min_samples=ms, metric="euclidean",
                                cluster_selection_method=sel, prediction_data=True)
        vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2),
                                           min_df=1, max_df=0.70)
        tm = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model,
                      vectorizer_model=vectorizer_model, top_n_words=top_n,
                      min_topic_size=mcs, verbose=False)
        topics, _ = tm.fit_transform(SUPP_TEXTS, embeddings=SUPP_EMB)
        words = []
        for tid in sorted(set(int(x) for x in topics)):
            if tid == -1:
                continue
            words.append([w for w, _ in tm.get_topic(tid)][:top_n])
        return words

    _lda_path = ARTIFACT_DIR / "task1_lda_topics.csv"
    lda_words = []
    if _lda_path.exists():
        _lda = pd.read_csv(_lda_path).sort_values(["topic", "rank"])
        lda_words = [g["term"].astype(str).tolist()[:10] for _, g in _lda.groupby("topic")]

    coherence_rows = []
    if lda_words:
        coherence_rows.append({"method": "LDA", "n_topics": len(lda_words), "c_v": _cv(lda_words)})
    for label, (mcs, ms, nn, sel) in [
        ("BERTopic prescribed (eom)", (25, 5, 20, "eom")),
        ("BERTopic leaf", (25, 5, 20, "leaf")),
    ]:
        tw = _bertopic_topic_words(mcs, ms, nn, sel)
        coherence_rows.append({"method": label, "n_topics": len(tw), "c_v": _cv(tw)})

    coherence_df = pd.DataFrame(coherence_rows)
    coherence_df.to_csv(ARTIFACT_DIR / "task1_modern_coherence.csv", index=False)
    display(coherence_df)
    for r in coherence_rows:
        print(f"{r['method']}: n_topics={r['n_topics']} c_v={r['c_v']:.4f}")


### Interpreting modern topic outputs

When interpreting results, compare target-category modern topics with n-grams and LDA —
all three answer the same thematic question. Mention fallback or outlier behavior only if it materially affects interpretation, and avoid describing fallback outputs as equivalent to full transformer-based BERTopic.

## 9. Compare the three approaches

This section is the core and most important intellectual component of the assignment. You are expected to go beyond simply running three algorithms. You should analyze what each method reveals and what it misses.

An good comparison will typically address:

1. **N-grams:** Provides transparent, surface-level phrases that are easy to interpret but struggle to group synonyms or represent broader concepts.
2. **LDA:** Produces compact latent themes that help summarize corpus-level structure, yet may blend unrelated ideas, especially in short or noisy reviews.
3. **BERTopic / modern clustering:** Often yields more semantically coherent topics and can uncover fine-grained clusters, but is more sensitive to hyperparameters, computational resources, and outliers.

You should also explicitly tie this analysis to later capstone tasks. For instance, dish-related terms surfaced here can guide Task 3 (phrase discovery), while contrasts between topics associated with high- vs. low-rated reviews can motivate Task 4 (evidence-grounded dish ranking).

In [ ]:
comparison_prompts = pd.DataFrame({
    "question": [
        "Which themes appear consistently across n-grams, LDA, and the modern method?",
        "Which themes appear only in one method? Why might that happen?",
        "What differs between high-rated and low-rated reviews?",
        "Which discovered terms or themes could be useful for Task 2 cuisine/category mapping?",
        "Which dish-like phrases could become seeds for Task 3 phrase/dish discovery?",
        "Which sentiment-bearing themes could matter for Task 4 dish-to-restaurant ranking?",
    ],
    "instructor_note": [
        "Look for repeated food/service/ambience terms across methods.",
        "N-grams may surface concrete dishes; LDA may surface broad aspects; BERTopic may surface clusters.",
        "Students should discuss aspects, not just positive/negative words.",
        "Similarity maps benefit from coherent category/cuisine signals.",
        "Dish names should be exported later as a curated vocabulary.",
        "Complaints/praise around dish mentions can be used as sentiment evidence.",
    ],
    "student_answer": [
        "All three methods center on core Mexican dining vocabulary - tacos, salsa, burrito, carne asada, and chips/salsa - visible in overall n-grams, LDA Topics 1/2/5/7, and BERTopic Topic 0 (97.6% of the modern sample). Service and dining-process language also recur, but with different granularity: n-gram tokens vs. refined LDA Topic 4 vs. BERTopic's broad dominant cluster.",
        "N-grams uniquely highlight raw rating-tone phrases (delicious, fresh, friendly vs. bad, minutes, asked) without aspect structure. LDA uniquely separates interpretable aspects: Topic 4 service/wait/process, Topic 3 bar/happy hour, Topic 1 authenticity/family, Topic 0 burrito/fast-casual, and Topics 5/7 named dishes. BERTopic alone surfaces a small Chipotle-specific cluster (Topic 1, 2.4%) because embeddings group brand-specific wording that LDA spreads across several topics.",
        "High vs. low is not just positive vs. negative adjectives. Refined LDA Topic 4 (time, order, minutes, asked, took, bad) is about 5x stronger in low reviews (0.404 vs. 0.077). High reviews emphasize aspects: authenticity/family dining (Topic 1), bar/social dining (Topic 3), and named dishes such as carne asada and fish tacos (Topics 5 and 7). N-grams echo this: low reviews elevate service/time/process terms; high reviews elevate food quality and specific items alongside positive evaluative words.",
        "For Task 2 cuisine/category mapping, stable anchors include mexican, authentic, salsa, chips salsa, family, and bar/happy-hour language from LDA Topics 1 and 3 plus recurring n-grams. These signal cuisine identity and dining style rather than one-off dish mentions.",
        "Task 3 phrase/dish seeds: carne asada, burrito, tacos, fish tacos, shrimp, pork, tortillas, chips salsa, and breakfast burrito phrases from LDA Topics 5 and 7 and high-rated n-grams. These are concrete multi-word items suitable for a curated dish vocabulary.",
        "For Task 4 ranking evidence, contrast service/wait complaints (refined LDA Topic 4; low n-grams such as minutes, asked, bad, and service/process terms) against dish- and ambiance-associated praise (Topics 1, 3, 5, and 7; high n-grams such as delicious, fresh, friendly). BERTopic confirms food themes but adds little rating discrimination here, so LDA + n-grams are stronger sentiment-aspect evidence.",
    ],
})
display(comparison_prompts)
comparison_prompts.to_csv(ARTIFACT_DIR / "task1_comparison_prompts.csv", index=False)


### Cross-method comparison - student answers

Answers below are grounded in saved artifacts (`theme_comparison_summary.csv`, `task1_lda_topics.csv`, `task1_modern_topic_diagnostics.csv`, and n-gram exports). See the `comparison_prompts` table above for the same text in CSV form.

1. **Consistent themes:** Mexican food vocabulary (tacos, salsa, burrito, carne asada, chips/salsa) appears in all three pipelines. Service and dining-process language also recur, but n-grams expose raw tokens, refined LDA splits a dedicated service/process topic (Topic 4), and BERTopic folds most reviews into one broad food cluster.

2. **Method-specific themes:** N-grams surface evaluative and complaint tokens (delicious, fresh, bad, minutes, asked) without latent structure. LDA yields distinct aspects: service/process (Topic 4), bar/social dining (Topic 3), authenticity/family dining (Topic 1), fast-casual/burrito language (Topic 0), and named dishes (Topics 5 and 7). BERTopic isolates a small Chipotle-specific cluster (Topic 1, 2.4%) that LDA dilutes across several topics.

3. **High vs. low (aspects, not just polarity):** The strongest signal is refined LDA Topic 4 (service/wait/process), which is much stronger in low-rated reviews (0.404 vs. 0.077). High-rated reviews emphasize authenticity/family dining (Topic 1), bar/social dining (Topic 3), and named dishes such as carne asada and fish tacos (Topics 5 and 7). N-grams align: low reviews rank service/time/process terms; high reviews mix food items with positive descriptors.

4. **Task 2 (cuisine mapping):** Use stable cuisine/style anchors - *mexican, authentic, salsa, family, chips salsa, bar/happy hour* - from LDA Topics 1 and 3 and frequent n-grams.

5. **Task 3 (dish seeds):** Export *carne asada, burrito, tacos, fish tacos, shrimp, pork, tortillas, chips salsa* from LDA Topics 5 and 7 and high-rated n-grams.

6. **Task 4 (ranking evidence):** Pair service/wait complaints (refined LDA Topic 4; low n-grams such as minutes, asked, bad, and service/process terms) with dish- and ambiance-linked praise (Topics 1, 3, 5, and 7; high n-grams). Treat BERTopic here as a model-health check because it has limited high/low separation at two topics.

**Trade-off summary:** N-grams are the most auditable baseline; refined LDA best quantifies aspect shifts by rating; BERTopic confirms semantic food themes but is too coarse in this run for fine-grained landscape mapping.


**Insights! 💡**

To extend the mental model of techniques used for topic modeling, text mining, and NLP across different “eras,” we can situate the methods in this notebook within a longer methodological lineage:

**The "Pre-History" (1950s – 1970s): Rule-Based & Symbolic AI**
- **The Tech:** Regular expressions, context-free grammars, hand-built lexicons, and manually curated ontologies.
- **The Vibe:** Everything was a strict “if-then” rule. If the word “not” appeared, flip the sentiment. There was no statistical learning—just linguistics, logic, and manually encoded knowledge.

**The Statistical Era (1980s – 2000s): TF, TF-IDF, and N-grams**
- **The Tech:** Term frequency, TF-IDF, bag-of-words vectors, n-grams, cosine similarity, and classical information retrieval.
- **The Vibe:** Text became countable. A document could be represented by the words and phrases it contained, weighted by how distinctive those terms were. This era gave us fast, interpretable baselines that are still useful today.
- **In this course:** Task 1 uses n-grams to identify frequent review themes. Later tasks reuse TF-IDF-style representations for similarity mapping, temporal term analysis, and prediction.

**The Latent Semantic Era (1990s – Early 2000s): Between TF-IDF and LDA**
- **The Tech:** LSA/LSI, or Latent Semantic Analysis / Latent Semantic Indexing, based on Singular Value Decomposition (SVD).
- **The Gap:** TF-IDF represented documents as sparse word-count vectors, but it did not directly capture hidden semantic structure. LSA used SVD to compress a term-document matrix into a smaller set of latent dimensions. This helped reveal that words such as “doctor” and “physician” ***may*** behave similarly because they appear in related contexts, even if they are not the same token.
- **The Vibe:** LSA was an early attempt to compress meaning. It was not yet a probabilistic topic model or a neural embedding model, but it served as an important bridge from raw word counts toward latent semantic representations.
- **In this course:** Task 1 focuses on n-grams, LDA, and BERTopic / embedding-based topic discovery. We revisit the LSA/SVD idea in Task 6, where Truncated SVD is used as a compact text representation for supervised prediction.

**The Probabilistic Topic Era (2000s – 2010s): LDA and Bayesian Topic Models**
- **The Tech:** pLSA, LDA, and related probabilistic topic models.
- **The Gap:** LSA compressed the term-document matrix, but its latent dimensions were not always easy to interpret probabilistically. LDA introduced a generative model: documents are mixtures of topics, and topics are distributions over words.
- **The Vibe:** Instead of simply counting words or compressing matrices, LDA asked: “What hidden topics could have generated these documents?” It became a standard classical tool for discovering themes in large text collections.
- **In this notebook:** LDA gives us a classical topic-modeling lens for understanding what people talk about in restaurant reviews.

**The Neural Transition (2010 – 2017): Word Vectors and Sequential Models**
- **The Tech:** Word2Vec, GloVe, RNNs, LSTMs, and early encoder-decoder models.
- **The Gap:** LDA gave interpretable topics, but it ignored word ***order*** and represented words mostly through topic co-occurrence.
  - Word2Vec and related models gave words dense vector representations, while RNNs and LSTMs tried to model sequences by reading text step by step.
- **The Vibe:** 
  - This era shifted NLP from sparse symbolic/statistical representations toward dense learned representations. It was the bridge from classical text mining to modern deep learning NLP. 
  - This middle era was obsessed with Sequential Processing—trying to read a sentence from left-to-right to build a hidden "memory" of the text, before Transformers (2017) made it all parallel.

**The Embedding / Transformer Era (2017 – Present): Contextual Meaning**
- **The Tech:** BERT, sentence-transformers, transformer sentiment models, BERTopic, and embedding-based clustering.
- **The Gap:** Earlier word vectors gave each word a mostly fixed meaning. Transformers made representations contextual: the meaning of a word or sentence ***depends on the surrounding text***.
- **The Vibe:** Instead of representing documents only through counts or topic mixtures, modern methods map text into dense semantic spaces. Similar ideas, reviews, cuisines, or dishes can be near each other even when they do not share the same exact vocabulary.
- **In this notebook:** BERTopic / embedding-based topic discovery gives us a modern contrast to n-grams and LDA. It helps show how semantic representations can surface themes that are not obvious from word counts alone.
- **Across the course:** Embedding-based representations return in Task 2 for cuisine/subcategory similarity mapping, in Task 3 through KeyBERT’s semantic keyphrase extraction, and in Task 4 through transformer-based sentiment over dish-local evidence. This lets you see embeddings not as a single tool, but as a reusable representation layer for topic discovery, similarity, phrase discovery, and sentiment analysis.

**The "Next Era" (2024 – Future): Agentic Reasoning and Evidence-Grounded Systems**
- **The Tech:** LLM agents, multimodal models, long-context models, retrieval-augmented generation (RAG), and tool-using systems.
- **The Vibe:** We are moving beyond static representations toward systems that retrieve evidence, interpret intent, execute tools, and maintain state over time. The important question is no longer just “What text is similar?” but “What evidence supports this claim, and can the system justify its reasoning?”
- **In this course:** We do not build a full RAG system as a required task, but Task 4 introduces the core idea behind RAG: recommendations and claims should be grounded in retrieved or selected evidence.

**The Timeline Summary:**
- Symbolic Era: logic, rules, grammars, and hand-built knowledge
- Statistical Era: TF, TF-IDF, n-grams, cosine similarity
- Latent Semantic Era: LSA/LSI and SVD-based semantic compression
- Probabilistic Topic Era: pLSA, LDA, and Bayesian topic models
- Neural Transition Era: Word2Vec, GloVe, RNNs, and LSTMs
- Embedding / Transformer Era: BERT, sentence-transformers, BERTopic, GPT-style models
- Agentic / Evidence-Grounded Era: RAG, tool use, reasoning, autonomy, and grounded claims

### Downstream handoff: topic and theme summaries

These summary exports give later tasks a compact view of Task 1 themes. They are intentionally descriptive artifacts: later notebooks may reuse them for category mapping, dish/entity seed ideas, evidence-grounded explanations, temporal monitoring, or prediction features, but the original Task 1 modeling outputs remain the source of record.

### Topic Summary Helper

This helper standardizes topic outputs from LDA and BERTopic into a uniform summary
table, making the Task 1 artifact easier to compare and reuse in later tasks.


In [ ]:
# Stable downstream topic/theme summaries.
topic_summary_frames = []


def append_topic_summary(topic_table, method_label, theme_prefix, prevalence_df=None):
    if topic_table is None or not isinstance(topic_table, pd.DataFrame) or topic_table.empty:
        return
    summary = (
        topic_table.sort_values(["topic", "rank"])
        .groupby("topic")
        .agg(
            top_terms=("term", lambda s: "; ".join(s.astype(str).head(12))),
            max_topic_term_weight=("weight", "max"),
        )
        .reset_index()
    )
    summary["method"] = method_label
    summary["theme_id"] = summary["topic"].map(lambda x: f"{theme_prefix}_{x}")
    if prevalence_df is not None and isinstance(prevalence_df, pd.DataFrame) and not prevalence_df.empty:
        prevalence_cols = [c for c in ["topic_num", "high", "low", "neutral"] if c in prevalence_df.columns]
        if "topic_num" in prevalence_cols:
            summary = summary.merge(prevalence_df[prevalence_cols], left_on="topic", right_on="topic_num", how="left")
    topic_summary_frames.append(summary.drop(columns=["topic", "topic_num"], errors="ignore"))

### Save Downstream Topic and Theme Summaries

These tables are compact handoff artifacts. They do not replace the detailed modeling outputs, but they give later tasks and grader review a stable summary of the evidence produced in Task 1.

In [ ]:
# TODO (Category D - Medium pipeline): Assemble the cross-method topic comparison summary.
# Estimated time: ~1 h
#
# This cell synthesizes all three approaches (n-grams, LDA, modern) into exportable summaries.
# It is intentionally idempotent: rerunning this cell rebuilds the summary tables from scratch
# instead of appending duplicate topic rows.

# Step 1 - Populate topic_summary with one row per method/topic.
topic_summary_frames = []

modern_method_label = (
    "BERTopic_target_category"
    if modern_target_method == "BERTopic"
    else "modern_fallback_target_category"
)

modern_prevalence_for_summary = pd.DataFrame()
if (
    "modern_target_crosstab" in globals()
    and modern_target_crosstab is not None
    and isinstance(modern_target_crosstab, pd.DataFrame)
    and not modern_target_crosstab.empty
):
    modern_prevalence_for_summary = modern_target_crosstab.copy()
    if "topic" not in modern_prevalence_for_summary.columns:
        modern_prevalence_for_summary = modern_prevalence_for_summary.reset_index()
    if "topic" in modern_prevalence_for_summary.columns:
        modern_prevalence_for_summary["topic_num"] = modern_prevalence_for_summary["topic"].astype(int)

append_topic_summary(
    topic_table=refined_lda_topics,
    method_label="LDA_restaurant_stopwords",
    theme_prefix="refined_lda_topic",
    prevalence_df=refined_avg_prevalence,
)
append_topic_summary(
    topic_table=modern_target_topic_terms,
    method_label=modern_method_label,
    theme_prefix="modern_target_topic",
    prevalence_df=modern_prevalence_for_summary,
)

if topic_summary_frames:
    topic_summary = pd.concat(topic_summary_frames, ignore_index=True)
else:
    topic_summary = pd.DataFrame(
        columns=["method", "theme_id", "top_terms", "max_topic_term_weight", "high", "low", "neutral"]
    )

# Guard against accidental duplicate rows if earlier cells were rerun out of order.
topic_summary = topic_summary.drop_duplicates(subset=["method", "theme_id"], keep="last")
preferred_column_order = [
    "method",
    "theme_id",
    "top_terms",
    "max_topic_term_weight",
    "high",
    "low",
    "neutral",
]
existing_preferred_columns = [c for c in preferred_column_order if c in topic_summary.columns]
extra_columns = [c for c in topic_summary.columns if c not in existing_preferred_columns]
topic_summary = topic_summary[existing_preferred_columns + extra_columns].reset_index(drop=True)
topic_summary.to_csv(ARTIFACT_DIR / "topic_summary.csv", index=False)
display(topic_summary)

# Step 2 - Build a full-vocabulary high-vs-low n-gram comparison.
def compute_full_ngram_high_low_differences(
    df,
    text_col="clean_text",
    top_each_side=4,
    ngram_range=(1, 2),
):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return pd.DataFrame(), pd.DataFrame()

    text_col_to_use = text_col
    if text_col_to_use not in df.columns and "cleaned_text" in df.columns:
        text_col_to_use = "cleaned_text"

    work = df.loc[df["rating_group"].isin(["high", "low"]), [text_col_to_use, "rating_group"]].copy()
    work[text_col_to_use] = work[text_col_to_use].fillna("").astype(str)
    work = work[work[text_col_to_use].str.len() > 0].reset_index(drop=True)
    if work.empty:
        return pd.DataFrame(), pd.DataFrame()

    min_df_value = globals().get("minimum_document_frequency", 3 if DATA_SOURCE == "smoke_test" else 20)
    vectorizer = CountVectorizer(
        stop_words="english",
        ngram_range=ngram_range,
        min_df=min_df_value,
        max_df=0.70,
        token_pattern=r"(?u)\b\w[\w']{1,}\b",
    )
    try:
        X = vectorizer.fit_transform(work[text_col_to_use])
    except ValueError:
        return pd.DataFrame(), pd.DataFrame()

    terms = np.array(vectorizer.get_feature_names_out())
    rating_values = work["rating_group"].to_numpy()
    high_counts = np.asarray(X[rating_values == "high"].sum(axis=0)).ravel().astype(float)
    low_counts = np.asarray(X[rating_values == "low"].sum(axis=0)).ravel().astype(float)

    def has_domain_stopword(ngram):
        return any(tok in DOMAIN_STOPWORDS for tok in str(ngram).split())

    full_tbl = pd.DataFrame({
        "ngram": terms,
        "high_count": high_counts,
        "low_count": low_counts,
    })
    full_tbl = full_tbl.loc[~full_tbl["ngram"].apply(has_domain_stopword)].copy()

    high_total = float(full_tbl["high_count"].sum())
    low_total = float(full_tbl["low_count"].sum())
    full_tbl["high_prevalence"] = full_tbl["high_count"] / high_total if high_total else 0.0
    full_tbl["low_prevalence"] = full_tbl["low_count"] / low_total if low_total else 0.0
    full_tbl["high_minus_low"] = full_tbl["high_prevalence"] - full_tbl["low_prevalence"]
    full_tbl["abs_high_minus_low"] = full_tbl["high_minus_low"].abs()

    summary_ngram_exclusions = {
        "great", "love", "best", "nice", "little", "try", "pretty", "really", "just",
        "got", "came", "went", "said", "know", "going", "people", "way", "did", "don",
        "mexican", "time",
    }
    candidate_tbl = full_tbl.loc[~full_tbl["ngram"].isin(summary_ngram_exclusions)].copy()
    high_rows = candidate_tbl.sort_values("high_minus_low", ascending=False).head(top_each_side)
    low_rows = candidate_tbl.sort_values("high_minus_low", ascending=True).head(top_each_side)
    selected = (
        pd.concat([high_rows, low_rows], ignore_index=True)
        .drop_duplicates(subset=["ngram"])
        .reset_index(drop=True)
    )
    return full_tbl.sort_values("abs_high_minus_low", ascending=False).reset_index(drop=True), selected

ngram_full_diff, selected_ngrams = compute_full_ngram_high_low_differences(comparison_df)
if not ngram_full_diff.empty:
    ngram_full_diff.to_csv(ARTIFACT_DIR / "task1_ngram_high_low_full_differences.csv", index=False)

# Step 3 - Assemble theme_comparison_summary.
theme_comparison_rows = []
top_terms_lookup = topic_summary.set_index("theme_id")["top_terms"].to_dict() if not topic_summary.empty else {}

if not selected_ngrams.empty:
    for _, row in selected_ngrams.iterrows():
        safe_id = re.sub(r"[^a-z0-9]+", "_", str(row["ngram"]).lower()).strip("_")
        theme_comparison_rows.append({
            "method": "NGRAM_high_low",
            "theme_id": f"ngram_{safe_id}",
            "comparison": "high_vs_low_relative_frequency_full_vocab",
            "high_prevalence": float(row["high_prevalence"]),
            "low_prevalence": float(row["low_prevalence"]),
            "high_minus_low": float(row["high_minus_low"]),
            "top_terms": str(row["ngram"]),
        })

if refined_avg_prevalence is not None and not refined_avg_prevalence.empty:
    for _, row in refined_avg_prevalence.iterrows():
        topic_num = int(row["topic_num"])
        theme_id = f"refined_lda_topic_{topic_num}"
        high = row.get("high", np.nan)
        low = row.get("low", np.nan)
        theme_comparison_rows.append({
            "method": "LDA_restaurant_stopwords",
            "theme_id": theme_id,
            "comparison": "high_vs_low_prevalence",
            "high_prevalence": high,
            "low_prevalence": low,
            "high_minus_low": float(high) - float(low) if pd.notna(high) and pd.notna(low) else np.nan,
            "top_terms": top_terms_lookup.get(theme_id, ""),
        })

if not modern_prevalence_for_summary.empty:
    for _, row in modern_prevalence_for_summary.iterrows():
        topic_num = int(row["topic_num"])
        theme_id = f"modern_target_topic_{topic_num}"
        high = row.get("high", np.nan)
        low = row.get("low", np.nan)
        theme_comparison_rows.append({
            "method": modern_method_label,
            "theme_id": theme_id,
            "comparison": "high_vs_low_prevalence",
            "high_prevalence": high,
            "low_prevalence": low,
            "high_minus_low": float(high) - float(low) if pd.notna(high) and pd.notna(low) else np.nan,
            "top_terms": top_terms_lookup.get(theme_id, ""),
        })

if "comparison_prompts" in globals() and isinstance(comparison_prompts, pd.DataFrame) and not comparison_prompts.empty:
    for prompt_i, row in comparison_prompts.reset_index(drop=True).iterrows():
        theme_comparison_rows.append({
            "method": "cross_method_prompt",
            "theme_id": f"prompt_{prompt_i + 1}",
            "comparison": "qualitative_cross_method_answer",
            "high_prevalence": np.nan,
            "low_prevalence": np.nan,
            "high_minus_low": np.nan,
            "top_terms": row.get("student_answer", ""),
            "question": row.get("question", ""),
            "instructor_note": row.get("instructor_note", ""),
            "student_answer": row.get("student_answer", ""),
        })

theme_comparison_summary = pd.DataFrame(theme_comparison_rows)
for col in ["question", "instructor_note", "student_answer"]:
    if col not in theme_comparison_summary.columns:
        theme_comparison_summary[col] = ""

method_order = {
    "NGRAM_high_low": 0,
    "LDA_restaurant_stopwords": 1,
    modern_method_label: 2,
    "cross_method_prompt": 3,
}
theme_comparison_summary["_method_order"] = theme_comparison_summary["method"].map(method_order).fillna(99)
theme_comparison_summary["_abs_diff"] = theme_comparison_summary["high_minus_low"].abs()
theme_comparison_summary = (
    theme_comparison_summary
    .sort_values(["_method_order", "_abs_diff"], ascending=[True, False], na_position="last")
    .drop(columns=["_method_order", "_abs_diff"])
    .reset_index(drop=True)
)

summary_column_order = [
    "method",
    "theme_id",
    "comparison",
    "high_prevalence",
    "low_prevalence",
    "high_minus_low",
    "top_terms",
    "question",
    "instructor_note",
    "student_answer",
]
theme_comparison_summary = theme_comparison_summary[
    [c for c in summary_column_order if c in theme_comparison_summary.columns]
    + [c for c in theme_comparison_summary.columns if c not in summary_column_order]
]
theme_comparison_summary.to_csv(ARTIFACT_DIR / "theme_comparison_summary.csv", index=False)
display(theme_comparison_summary)

if refined_lda_topics is not None and isinstance(refined_lda_topics, pd.DataFrame) and not refined_lda_topics.empty:
    refined_lda_topics.to_csv(ARTIFACT_DIR / "task1_lda_topics.csv", index=False)
    print("Saved canonical LDA artifact from refined_lda_topics")

# --- Validation ---
assert topic_summary is not None
assert topic_summary.duplicated(subset=["method", "theme_id"]).sum() == 0
assert (ARTIFACT_DIR / "topic_summary.csv").exists()
assert (ARTIFACT_DIR / "theme_comparison_summary.csv").exists()
assert (ARTIFACT_DIR / "task1_lda_topics.csv").exists()
assert "cross_method_prompt" in set(theme_comparison_summary["method"])
print(f"✓ Cross-method summary: {len(topic_summary)} topic rows, {len(theme_comparison_summary)} comparison rows")


### Comparison writing guidance

Your comparison should explain trade-offs, not declare a single winner. Highlight where n-grams are transparent but shallow, where LDA is interpretable but mixed, and where modern methods improve semantic grouping but add complexity and model-health caveats.


## Visual Evidence and Interpretation

Visualizations in this notebook are intended to function as evidence rather than decoration. Each figure should directly support one or more goals: interpreting results, comparing methods, conducting diagnostic analyses, or communicating discovered Yelp restaurant-intelligence patterns.

In your student report, for each selected figure, describe what it reveals and discuss at least one limitation of the visualization.

### Visualization Helpers

The plotting helpers provide consistent figure design: they bound labels and topic counts, save figures as PNGs, and, when useful, generate corresponding CSV files.

In [ ]:
# Visual evidence layer: LDA heatmaps, rating-group prevalence, and optional topic networks.
import textwrap
VIS_TOP_TERMS = 24
VIS_MAX_TOPICS = 8
VIS_TERMS_PER_TOPIC = 8


def _short_label(value, width=22):
    return "\n".join(textwrap.wrap(str(value), width=width)) or str(value)


def _plot_heatmap_dataframe(df: pd.DataFrame, title: str, filename: str, cbar_label: str = "value"):
    if df is None or df.empty or df.shape[0] == 0 or df.shape[1] == 0:
        print(f"Skipping {filename}: not enough data for a heatmap.")
        return False
    fig_w = max(7, min(15, 0.55 * len(df.columns) + 3))
    fig_h = max(3.5, min(10, 0.45 * len(df.index) + 2.5))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(df.values, aspect="auto")
    ax.set_xticks(np.arange(len(df.columns)))
    ax.set_xticklabels([_short_label(c, 16) for c in df.columns], rotation=45, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(df.index)))
    ax.set_yticklabels([_short_label(i, 18) for i in df.index], fontsize=9)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=cbar_label, fraction=0.046, pad=0.04)
    fig.tight_layout()
    save_current_figure(filename)
    plt.show()
    return True


def write_topic_term_heatmap(topic_table, title, filename, values_filename, label_prefix="Topic"):
    if topic_table is None or not isinstance(topic_table, pd.DataFrame) or topic_table.empty:
        print(f"Skipping {filename}: topic table is unavailable or empty.")
        return None
    topic_col, term_col, weight_col = "topic", "term", "weight"
    selected_topics = [t for t in sorted(topic_table[topic_col].dropna().unique()) if int(t) != -1][:VIS_MAX_TOPICS]
    if not selected_topics:
        print(f"Skipping {filename}: no non-outlier topics available.")
        return None
    top_topic_terms = (topic_table[topic_table[topic_col].isin(selected_topics)]
        .sort_values([topic_col, weight_col], ascending=[True, False])
        .groupby(topic_col).head(VIS_TERMS_PER_TOPIC))
    selected_terms = (top_topic_terms.groupby(term_col)[weight_col].max()
        .sort_values(ascending=False).head(VIS_TOP_TERMS).index.tolist())
    heat = (topic_table[topic_table[topic_col].isin(selected_topics) & topic_table[term_col].isin(selected_terms)]
        .pivot_table(index=topic_col, columns=term_col, values=weight_col, fill_value=0)
        .reindex(index=selected_topics, columns=selected_terms, fill_value=0))
    heat.index = [f"{label_prefix} {t}" for t in heat.index]
    heat.to_csv(ARTIFACT_DIR / values_filename)
    _plot_heatmap_dataframe(heat, title, filename, "topic-term weight")
    return top_topic_terms

### Modern Topic Heatmap

The heatmap converts modern topic outputs into visual evidence for your report comparison.

In [ ]:
write_topic_term_heatmap(
    modern_target_topic_terms,
    "Target-category modern topic-term evidence heatmap",
    "modern_topic_heatmap_target.png",
    "task1_modern_topic_heatmap_target_terms.csv",
    label_prefix="Modern Topic",
)

## 10. Required Report Deliverables

Your PDF report should be approximately **2–3 pages** and include:

- **Subset definition**: cuisine/category focus, sample counts, high vs. low split rationale.
- **EDA evidence**: at least one plot showing review distribution or rating patterns.
- **Frequency analysis**: top n-grams from high and low reviews; what themes do they reveal?
- **LDA findings**: key topics from the domain-specific LDA model; how do topics differ by rating?
- **BERTopic/modern findings**: modern topics discovered; where do they agree or disagree with LDA?
- **Three-method comparison**: where do frequency, LDA, and modern topics agree? Where do they
  disagree? What does each method capture that the others miss?

### Final deliverable checklist

Your report should include: subset definition, EDA evidence, n-gram analysis,
LDA topic modeling results (with brief interpretation), modern embedding/topic
analysis, a cross-method comparison, and a written interpretation grounded in
saved artifacts.


### Common pitfalls to avoid

- Using an undersized subset without discussing limitations.
- Showing plots/tables without interpretation.
- Treating unsupervised topics as ground truth labels.
- Skipping high-vs-low comparison logic.
- Treating modern-topic diagnostics as substantive themes rather than model-health checks.
- Omitting reproducibility details (sampling rules, seeds, stopword choices, package path).


## Final artifact validation

Run this cell before submission. It checks that required artifacts exist, are non-empty, and satisfy the most important schema expectations. Optional artifacts are reported but do not block submission.

In [ ]:
# Final artifact listing
print("Artifacts written to:", ARTIFACT_DIR.resolve())
expected_artifacts = [
    "review_subset.csv",
    "task1_overall_top_ngrams.csv",
    "task1_lda_topics.csv",
    "task1_modern_topic_diagnostics.csv",
    "theme_comparison_summary.csv",
]
missing = [f for f in expected_artifacts if not (ARTIFACT_DIR / f).exists()]
if missing:
    print("⚠  Missing artifacts:", missing)
else:
    print("✓ All required artifacts present.")
for f in expected_artifacts:
    p = ARTIFACT_DIR / f
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status}: {f}")